In [30]:
import pandas as pd
import numpy as np
import random
import re
import unicodedata
from datetime import datetime, timedelta
from faker import Faker
from geopy.distance import great_circle
from urllib import parse
from sqlalchemy import create_engine, text
from math import floor
from collections import defaultdict

fake = Faker()

## Global Settings

If the tables already exist in csv-form and no changes are required, skip directly to the end. [Loading Option II: from CSVs](#loading-option-ii-from-csvs)

In [25]:
# choose output: "sql" or "excel"
simulation_output = "sql"

csv_folder_path = ""    # e.g. "D:/airline_csvs/"
xlsx_folder_path = ""

# choose csv separator and decimal format
separator = ";"
dec = ","

# time frame for SQL database; significant impact on required storage space and performance
start_date_sql, end_date_sql = datetime(2022, 1, 1), datetime(2025, 1, 1)

# time frame Excel snapshot; a much shorter period is recommended due to Excel performance limitations
start_date_excel, end_date_excel = datetime(2024, 4, 8), datetime(2024, 4, 29)

if simulation_output == "sql":
    start_date_global, end_date_global = start_date_sql, end_date_sql
elif simulation_output == "excel":
    start_date_global, end_date_global = start_date_excel, end_date_excel
else:
    raise ValueError("simulation_output must be either 'sql' or 'excel'")


sim_days_global = (end_date_global - start_date_global).days
print(f"start: {start_date_global}, end: {end_date_global}, days: {sim_days_global}")

start: 2022-01-01 00:00:00, end: 2025-01-01 00:00:00, days: 1096


In [26]:
# Note: For populating a SQL database csv-files are not required, tables can be loaded into the database directly from dfs.
#       If the unmodified tables are supposed to also be used in Excel or Power BI, all csv-files can be stored at the end (code provided).
#       If the population code is being actively worked on over the course of multiple sessions, it is recommended to create a backup-csv
#       after each table is finalized (code provided).

# csv settings:
# "end" for storing all tables as csvs at the end |
# "imm" for immediate storage after each table is finalized |
# "" (or any other value) for no csv-files
csv_setting = "end"

if simulation_output == "sql":
    csv_settings = csv_setting
else: csv_settings = "no"

In [29]:
# create (empty) SQL tables (to populate with dfs) in this notebook or from external .sql-script (user preference, functionally equivalent)

database_tables = "here"    # "here" for this notebook | "script" (or any other value) if external .sql-script is used

# Create Airline Data

## Airports Table

#### Assuming Frankfurt as primary Hub, London and Paris as secondary hubs

In [6]:
# Airports for short-haul flights
# should account for >65% of routes; target: ~80 routes, 2-3 flights per route per day, 160-180 total flights per day, ~35 aircraft needed
airports_data_short = [
    {"airport_code": "FRA", "airport_name": "Frankfurt Airport", "city": "Frankfurt", "country": "Germany",
     "coords": (8.5622, 50.0379), "climate_region": "Temperate"},
    {"airport_code": "LHR", "airport_name": "London Heathrow Airport", "city": "London", "country": "United Kingdom",
     "coords": (-0.4543, 51.4700), "climate_region": "Temperate"},
    {"airport_code": "BER", "airport_name": "Berlin Brandenburg Airport", "city": "Berlin", "country": "Germany",
     "coords": (13.5033, 52.3667), "climate_region": "Temperate"},
    {"airport_code": "HAM", "airport_name": "Hamburg Airport", "city": "Hamburg", "country": "Germany",
     "coords": (9.9882, 53.6304), "climate_region": "Temperate"},
    {"airport_code": "VIE", "airport_name": "Vienna International Airport", "city": "Vienna", "country": "Austria",
     "coords": (16.5697, 48.1103), "climate_region": "Temperate"},
    {"airport_code": "MUC", "airport_name": "Munich Airport", "city": "Munich", "country": "Germany",
     "coords": (11.7861, 48.3538), "climate_region": "Temperate"},
    {"airport_code": "MAD", "airport_name": "Adolfo Suárez Madrid–Barajas Airport", "city": "Madrid",
     "country": "Spain", "coords": (-3.5608, 40.4722), "climate_region": "Mediterranean"},
    {"airport_code": "PMI", "airport_name": "Palma de Mallorca Airport", "city": "Palma", "country": "Spain",
     "coords": (2.7388, 39.5517), "climate_region": "Mediterranean"},
    {"airport_code": "IST", "airport_name": "Istanbul Airport", "city": "Istanbul", "country": "Turkey",
     "coords": (28.7519, 41.2753), "climate_region": "Temperate"},
    {"airport_code": "BCN", "airport_name": "Barcelona–El Prat Airport", "city": "Barcelona", "country": "Spain",
     "coords": (2.0833, 41.2974), "climate_region": "Mediterranean"},
    {"airport_code": "CDG", "airport_name": "Charles de Gaulle Airport", "city": "Paris", "country": "France",
     "coords": (2.5479, 49.0097), "climate_region": "Temperate"},
    {"airport_code": "LIS", "airport_name": "Humberto Delgado Airport", "city": "Lisbon", "country": "Portugal",
     "coords": (-9.1342, 38.7742), "climate_region": "Mediterranean"},
    {"airport_code": "AMS", "airport_name": "Amsterdam Schiphol Airport", "city": "Amsterdam", "country": "Netherlands",
     "coords": (4.7683, 52.3105), "climate_region": "Temperate"},
    {"airport_code": "ZRH", "airport_name": "Zurich Airport", "city": "Zurich", "country": "Switzerland",
     "coords": (8.5555, 47.4581), "climate_region": "Temperate"},
    {"airport_code": "CPH", "airport_name": "Copenhagen Airport", "city": "Copenhagen", "country": "Denmark",
     "coords": (12.6561, 55.6181), "climate_region": "Temperate"},
    {"airport_code": "FCO", "airport_name": "Leonardo da Vinci–Fiumicino Airport", "city": "Rome", "country": "Italy",
     "coords": (12.2389, 41.8003), "climate_region": "Mediterranean"},
    {"airport_code": "ARN", "airport_name": "Stockholm Arlanda Airport", "city": "Stockholm", "country": "Sweden",
     "coords": (17.9186, 59.6519), "climate_region": "Cold"},
    {"airport_code": "HEL", "airport_name": "Helsinki-Vantaa Airport", "city": "Helsinki", "country": "Finland",
     "coords": (24.9633, 60.3172), "climate_region": "Cold"},
    {"airport_code": "OSL", "airport_name": "Oslo Gardermoen Airport", "city": "Oslo", "country": "Norway",
     "coords": (11.1004, 60.1939), "climate_region": "Cold"},
    {"airport_code": "WAW", "airport_name": "Warsaw Chopin Airport", "city": "Warsaw", "country": "Poland",
     "coords": (20.9671, 52.1657), "climate_region": "Cold"},
    {"airport_code": "ATH", "airport_name": "Athens International Airport", "city": "Athens", "country": "Greece",
     "coords": (23.9475, 37.9364), "climate_region": "Mediterranean"},
    {"airport_code": "BRU", "airport_name": "Brussels Airport", "city": "Brussels", "country": "Belgium",
     "coords": (4.4844, 50.9014), "climate_region": "Temperate"}
]

df_airports_short = pd.DataFrame(airports_data_short)
#df_airports_short

In [7]:
# Airports for medium-haul flights
# should account for ~25% of routes; target: ~32 routes, 2 flights per route per day, ~64 total flights per day, ~18 aircraft needed
airports_data_medium = [
    {"airport_code": "FRA", "airport_name": "Frankfurt Airport", "city": "Frankfurt", "country": "Germany",
     "coords": (8.5622, 50.0379), "climate_region": "Temperate"},
    {"airport_code": "LHR", "airport_name": "London Heathrow Airport", "city": "London", "country": "United Kingdom",
     "coords": (-0.4543, 51.4700), "climate_region": "Temperate"},
    {"airport_code": "CDG", "airport_name": "Charles de Gaulle Airport", "city": "Paris", "country": "France",
     "coords": (2.5479, 49.0097), "climate_region": "Temperate"},
    {"airport_code": "DXB", "airport_name": "Dubai International Airport", "city": "Dubai",
     "country": "United Arab Emirates", "coords": (55.3657, 25.2532), "climate_region": "Desert"},
    {"airport_code": "DOH", "airport_name": "Hamad International Airport", "city": "Doha", "country": "Qatar",
     "coords": (51.6081, 25.2736), "climate_region": "Desert"},
    {"airport_code": "CAI", "airport_name": "Cairo International Airport", "city": "Cairo", "country": "Egypt",
     "coords": (31.4056, 30.1219), "climate_region": "Desert"},
    {"airport_code": "TLV", "airport_name": "Ben Gurion Airport", "city": "Tel Aviv", "country": "Israel",
     "coords": (34.8867, 32.0114), "climate_region": "Mediterranean"},
    {"airport_code": "BEY", "airport_name": "Beirut–Rafic Hariri International Airport", "city": "Beirut",
     "country": "Lebanon", "coords": (35.4884, 33.8209), "climate_region": "Mediterranean"},
    {"airport_code": "AMM", "airport_name": "Queen Alia International Airport", "city": "Amman", "country": "Jordan",
     "coords": (35.9932, 31.7226), "climate_region": "Desert"},
    {"airport_code": "ALG", "airport_name": "Houari Boumediene Airport", "city": "Algiers", "country": "Algeria",
     "coords": (3.2154, 36.6910), "climate_region": "Mediterranean"},
    {"airport_code": "CMN", "airport_name": "Mohammed V International Airport", "city": "Casablanca",
     "country": "Morocco", "coords": (-7.5899, 33.3675), "climate_region": "Mediterranean"},
    {"airport_code": "TUN", "airport_name": "Tunis–Carthage International Airport", "city": "Tunis",
     "country": "Tunisia", "coords": (10.2272, 36.8510), "climate_region": "Mediterranean"},
    {"airport_code": "LCA", "airport_name": "Larnaca International Airport", "city": "Larnaca", "country": "Cyprus",
     "coords": (33.6249, 34.8751), "climate_region": "Mediterranean"}
]

df_airports_medium = pd.DataFrame(airports_data_medium)
#df_airports_medium

In [8]:
# Airports for long-haul flights
# should account for ~7.5% of routes; target: ~20 routes, 1 flight per route per day, ~20 total flights per day, ~11 aircraft needed
airports_data_long = [
    {"airport_code": "FRA", "airport_name": "Frankfurt Airport", "city": "Frankfurt", "country": "Germany",
     "coords": (8.5622, 50.0379), "climate_region": "Temperate"},
    {"airport_code": "JFK", "airport_name": "John F. Kennedy International Airport", "city": "New York",
     "country": "United States", "coords": (-73.7781, 40.6413), "climate_region": "Temperate"},
    {"airport_code": "SFO", "airport_name": "San Francisco International Airport", "city": "San Francisco",
     "country": "United States", "coords": (-122.3790, 37.6213), "climate_region": "Mediterranean"},
    {"airport_code": "YYZ", "airport_name": "Toronto Pearson International Airport", "city": "Toronto",
     "country": "Canada", "coords": (-79.6248, 43.6777), "climate_region": "Cold"},
    {"airport_code": "HKG", "airport_name": "Hong Kong International Airport", "city": "Hong Kong", "country": "China",
     "coords": (113.9185, 22.3080), "climate_region": "Subtropical"},
    {"airport_code": "SIN", "airport_name": "Singapore Changi Airport", "city": "Singapore", "country": "Singapore",
     "coords": (103.9915, 1.3644), "climate_region": "Tropical"},
    {"airport_code": "ICN", "airport_name": "Incheon International Airport", "city": "Seoul", "country": "South Korea",
     "coords": (126.4407, 37.4602), "climate_region": "Temperate"},
    {"airport_code": "GRU", "airport_name": "São Paulo/Guarulhos–Governador André Franco Montoro International Airport",
     "city": "São Paulo", "country": "Brazil", "coords": (-46.4731, -23.4356), "climate_region": "Subtropical"},
    {"airport_code": "NRT", "airport_name": "Narita International Airport", "city": "Tokyo", "country": "Japan",
     "coords": (140.3929, 35.7720), "climate_region": "Temperate"},
    {"airport_code": "JNB", "airport_name": "O. R. Tambo International Airport", "city": "Johannesburg",
     "country": "South Africa", "coords": (28.2420, -26.1337), "climate_region": "Subtropical"},
    {"airport_code": "SYD", "airport_name": "Sydney Kingsford Smith Airport", "city": "Sydney", "country": "Australia",
     "coords": (151.1753, -33.9399), "climate_region": "Subtropical"}
]

df_airports_long = pd.DataFrame(airports_data_long)
#df_airports_long

In [9]:
# combine airports dfs
df_airports = pd.concat(
    [df_airports_short.copy(), df_airports_medium.copy(), df_airports_long.copy()]).drop_duplicates().sort_values(
    "airport_code").reset_index(drop=True).rename(columns={"coords": "coordinates"})

# reformat coordinates to be recognized as type POINT in SQL
df_airports["coordinates"] = df_airports["coordinates"].apply(
    lambda coord: f'({coord[0]},{coord[1]})' if isinstance(coord, tuple) else coord)

def assign_hub_status(code_):
    if code_ == "FRA":
        return "primary"
    elif code_ in ["LHR", "CDG"]:
        return "secondary"
    else:
        return "none"

df_airports["hub_status"] = df_airports["airport_code"].apply(assign_hub_status)

df_airports

,airport_code,airport_name,city,country,coordinates,climate_region,hub_status
0,ALG,Houari Boumediene Airport,Algiers,Algeria,"(3.2154,36.691)",Mediterranean,none
1,AMM,Queen Alia International Airport,Amman,Jordan,"(35.9932,31.7226)",Desert,none
2,AMS,Amsterdam Schiphol Airport,Amsterdam,Netherlands,"(4.7683,52.3105)",Temperate,none
3,ARN,Stockholm Arlanda Airport,Stockholm,Sweden,"(17.9186,59.6519)",Cold,none
4,ATH,Athens International Airport,Athens,Greece,"(23.9475,37.9364)",Mediterranean,none
5,BCN,Barcelona–El Prat Airport,Barcelona,Spain,"(2.0833,41.2974)",Mediterranean,none
6,BER,Berlin Brandenburg Airport,Berlin,Germany,"(13.5033,52.3667)",Temperate,none
7,BEY,Beirut–Rafic Hariri International Airport,Beirut,Lebanon,"(35.4884,33.8209)",Mediterranean,none
8,BRU,Brussels Airport,Brussels,Belgium,"(4.4844,50.9014)",Temperate,none
9,CAI,Cairo International Airport,Cairo,Egypt,"(31.4056,30.1219)",Desert,none


In [27]:
if csv_settings == "imm":
    df_airports.to_csv(csv_folder_path + "airports.csv", index=False, sep=separator, decimal=dec)

## Aircraft Table

In [11]:
# 30-35 aircraft needed to serve 160 short-haul flights per day

aircraft_specs_short = [
    {"model": "MC-21-300", "manufacturer": "Irkut", "seat_capacity": 211, "range_km": 6000},
    {"model": "A320neo", "manufacturer": "Airbus", "seat_capacity": 180, "range_km": 6300},
    {"model": "E195-E2", "manufacturer": "Embraer", "seat_capacity": 132, "range_km": 4815},
    {"model": "CRJ900", "manufacturer": "Bombardier", "seat_capacity": 90, "range_km": 2956},
    {"model": "A220-300", "manufacturer": "Airbus", "seat_capacity": 145, "range_km": 6200}
]

aircraft_distrib_short = [
    (aircraft_specs_short[0], 9),   # MC-21-300
    (aircraft_specs_short[1], 10),  # A320neo
    (aircraft_specs_short[2], 5),   # E195-E2
    (aircraft_specs_short[3], 4),   # CRJ900
    (aircraft_specs_short[4], 8)    # A220-300
]

aircraft_instances_short = [spec for spec, num in aircraft_distrib_short for _ in range(num)]

df_aircraft_short = pd.DataFrame(aircraft_instances_short)

random.seed(50)
manufacture_years_short = [year for year in range(2016, 2021)]
df_aircraft_short["manufacture_year"] = random.choices(manufacture_years_short, k=len(aircraft_instances_short))

df_aircraft_short.insert(0, "aircraft_id", "AC" + (df_aircraft_short.index + 1000).astype(str))
len(df_aircraft_short)

36

In [12]:
# split aircraft into different pools for flight scheduling
df_parts = []

for model, group in df_aircraft_short.groupby("model"):
    group_shuffled = group.sample(frac=1, random_state=50)  # Shuffle within each group for randomness
    half = len(group_shuffled) // 2
    df_parts.append((group_shuffled.iloc[:half], group_shuffled.iloc[half:]))

df_aircraft_short_a = pd.concat([part[0] for part in df_parts]).reset_index(drop=True)
df_aircraft_short_b = pd.concat([part[1] for part in df_parts]).reset_index(drop=True)

print("len a:", len(df_aircraft_short_a), "len b:", len(df_aircraft_short_b))

len a: 17 len b: 19


In [13]:
# ~19 aircraft needed to serve 64 medium-haul flights per day

aircraft_specs_medium = [
    {"model": "737 MAX 8", "manufacturer": "Boeing", "seat_capacity": 189, "range_km": 6570},
    {"model": "737 MAX 9", "manufacturer": "Boeing", "seat_capacity": 193, "range_km": 6575},
    {"model": "A321neo", "manufacturer": "Airbus", "seat_capacity": 206, "range_km": 7400}
]
manufacture_years_medium = [year for year in range(2016, 2024)]

aircraft_distrib_medium = [
    (aircraft_specs_medium[0], 6),   # 737 MAX 8
    (aircraft_specs_medium[1], 7),   # 737 MAX 9
    (aircraft_specs_medium[2], 7),   # A321neo
]

aircraft_instances_medium = [spec for spec, num in aircraft_distrib_medium for _ in range(num)]

df_aircraft_medium = pd.DataFrame(aircraft_instances_medium)

random.seed(50)
df_aircraft_medium["manufacture_year"] = random.choices(manufacture_years_medium, k=len(aircraft_instances_medium))
df_aircraft_medium.insert(0, "aircraft_id", "AC" + (df_aircraft_medium.index + 2000).astype(str))
len(df_aircraft_medium)

20

In [14]:
# split aircraft into different pools for flight scheduling

df_parts = []

for model, group in df_aircraft_medium.groupby("model"):
    group_shuffled = group.sample(frac=1, random_state=50)  # Shuffle within each group for randomness
    half = len(group_shuffled) // 2
    df_parts.append((group_shuffled.iloc[:half], group_shuffled.iloc[half:]))

df_aircraft_medium_a = pd.concat([part[0] for part in df_parts]).reset_index(drop=True)
df_aircraft_medium_b = pd.concat([part[1] for part in df_parts]).reset_index(drop=True)

print("len a:", len(df_aircraft_medium_a), "len b:", len(df_aircraft_medium_b))

len a: 9 len b: 11


In [15]:
# ~11 aircraft needed to serve 20 long-haul flights per day

aircraft_specs_long = [
    {"model": "A340-500", "manufacturer": "Airbus", "seat_capacity": 270, "range_km": 16670},
    {"model": "777-200LR", "manufacturer": "Boeing", "seat_capacity": 317, "range_km": 17395}
]

aircraft_distrib_long = [
    (aircraft_specs_long[0], 5),   # A340-500
    (aircraft_specs_long[1], 7),   # 777-200LR
]

aircraft_instances_long = [spec for spec, num in aircraft_distrib_long for _ in range(num)]

df_aircraft_long = pd.DataFrame(aircraft_instances_long)

random.seed(50)
manufacture_years_long = [year for year in range(2007, 2011)]
df_aircraft_long["manufacture_year"] = random.choices(manufacture_years_long, k=len(aircraft_instances_long))

df_aircraft_long.insert(0, "aircraft_id", "AC" + (df_aircraft_long.index + 3000).astype(str))
len(df_aircraft_long)

12

In [16]:
# spec check

if not df_aircraft_medium["range_km"].min() > df_aircraft_short["range_km"].max():
    raise ValueError
if not df_aircraft_long["range_km"].min() > df_aircraft_medium["range_km"].max():
    raise ValueError

In [17]:
# final table

df_aircraft = pd.concat([df_aircraft_short.copy(), df_aircraft_medium.copy(), df_aircraft_long.copy()]).sort_values(
    "aircraft_id").reset_index(drop=True)
df_aircraft

,aircraft_id,model,manufacturer,seat_capacity,range_km,manufacture_year
0,AC1000,MC-21-300,Irkut,211,6000,2018
1,AC1001,MC-21-300,Irkut,211,6000,2017
2,AC1002,MC-21-300,Irkut,211,6000,2019
3,AC1003,MC-21-300,Irkut,211,6000,2017
4,AC1004,MC-21-300,Irkut,211,6000,2018
...,...,...,...,...,...,...
63,AC3007,777-200LR,Boeing,317,17395,2008
64,AC3008,777-200LR,Boeing,317,17395,2009
65,AC3009,777-200LR,Boeing,317,17395,2007
66,AC3010,777-200LR,Boeing,317,17395,2007


In [11]:
if csv_settings == "imm":
    df_aircraft.to_csv(csv_folder_path + "aircraft.csv", index=False, sep=separator, decimal=dec)

## Routes Table

In [18]:
def generate_all_possible_routes(airports_data):
    # Extract coordinates
    airport_coords = {
        airport["airport_code"]: airport["coords"]
        for airport in airports_data
    }

    # Create coordinates DataFrame
    df_coords = pd.DataFrame([
        {"airport_code": code_, "lon": lon, "lat": lat}
        for code_, (lon, lat) in airport_coords.items()
    ])

    # Generate all valid route pairs (excluding self-pairs)
    route_pairs = [
        (a1["airport_code"], a2["airport_code"])
        for i_, a1 in df_coords.iterrows()
        for j, a2 in df_coords.iterrows()
        if i_ != j
    ]

    # Create route data with distances
    all_route_data = []
    for dep, arr in route_pairs:
        dep_coords = df_coords.loc[df_coords["airport_code"] == dep, ["lat", "lon"]].values[0]
        arr_coords = df_coords.loc[df_coords["airport_code"] == arr, ["lat", "lon"]].values[0]
        dist_km = round(great_circle(dep_coords, arr_coords).km)
        all_route_data.append({
            "departure_airport_code": dep,
            "arrival_airport_code": arr,
            "distance_km": dist_km
        })

    return pd.DataFrame(all_route_data)

### Short-Haul Routes
#### Intra-European Flights, mostly under 1500 km

In [19]:
# manual route selection by user preference

df_all_routes_short = generate_all_possible_routes(airports_data_short)

df_short_routes_fra = df_all_routes_short.query("departure_airport_code == 'FRA' or arrival_airport_code == 'FRA'")

df_short_routes_lhr = df_all_routes_short.query("(departure_airport_code != 'FRA' and arrival_airport_code != 'FRA')"
                                                " and (departure_airport_code == 'LHR' or arrival_airport_code == 'LHR')"
                                                ).iloc[np.r_[0:3, 19:23, 39]]

df_short_routes_indices = df_short_routes_fra.index.union(df_short_routes_lhr.index)

df_short_routes_cdg = df_all_routes_short.drop(df_short_routes_indices).query(
                      "departure_airport_code == 'CDG' or arrival_airport_code == 'CDG'"
                      ).iloc[np.r_[0:3, 9:12, 28, 39]]

df_short_routes_indices = df_short_routes_indices.union(df_short_routes_cdg.index)

df_short_routes_vie = df_all_routes_short.drop(df_short_routes_indices).query(
                      "departure_airport_code == 'VIE' or arrival_airport_code == 'VIE'"
                      ).iloc[np.r_[0:2, 2:5, 20, 21, 37]]

df_short_routes_indices = df_short_routes_indices.union(df_short_routes_vie.index)

df_short_routes_mad = df_all_routes_short.drop(df_short_routes_indices).query(
                      "departure_airport_code == 'MAD' or arrival_airport_code == 'MAD'"
                      ).iloc[np.r_[0:2, 3, 5:7, 8, 24, 39]]

df_short_routes_indices = df_short_routes_indices.union(df_short_routes_mad.index)

df_short_routes_ist = df_all_routes_short.drop(df_short_routes_indices).query(
                      "departure_airport_code == 'IST' or arrival_airport_code == 'IST'"
                      ).iloc[np.r_[0:2, 3, 7:9, 10]]

df_short_routes_indices = df_short_routes_indices.union(df_short_routes_ist.index)

df_short_routes_arn = df_all_routes_short.drop(df_short_routes_indices).query(
                      "departure_airport_code == 'ARN' or arrival_airport_code == 'ARN'"
                      ).iloc[np.r_[0:2, 15:17]]

df_short_routes_indices = df_short_routes_indices.union(df_short_routes_arn.index)

df_short_routes_ber = df_all_routes_short.drop(df_short_routes_indices).query(
                      "departure_airport_code == 'BER' or arrival_airport_code == 'BER'"
                      ).iloc[np.r_[0:2, 14:16]]

df_short_routes_indices = df_short_routes_indices.union(df_short_routes_ber.index)

df_short_routes_muc = df_all_routes_short.drop(df_short_routes_indices).query(
                      "departure_airport_code == 'MUC' or arrival_airport_code == 'MUC'"
                      ).iloc[np.r_[1, 3]]

df_short_routes_indices = df_short_routes_indices.union(df_short_routes_muc.index)
#df_all_routes_short.iloc[df_short_routes_indices]

In [20]:
df_routes_short = df_all_routes_short.copy().iloc[df_short_routes_indices]

df_routes_short["flights_per_day"] = 2

df_routes_short.reset_index(drop=True, inplace=True)
df_routes_short.insert(0, "line_number", 101 + df_routes_short.index)

df_routes_short

,line_number,departure_airport_code,arrival_airport_code,distance_km,flights_per_day
0,101,FRA,LHR,654,2
1,102,FRA,BER,431,2
2,103,FRA,HAM,411,2
3,104,FRA,VIE,621,2
4,105,FRA,MUC,300,2
...,...,...,...,...,...
85,186,BRU,FRA,304,2
86,187,BRU,LHR,350,2
87,188,BRU,VIE,925,2
88,189,BRU,MAD,1316,2


In [17]:
df_routes_short.flights_per_day.sum()

180

### Medium-Haul Routes
#### Flights between Europe and the Middle East, mostly between 1500 and 5000 km

In [23]:
# manual route selection by user preference

df_all_routes_medium = generate_all_possible_routes(airports_data_medium)

df_medium_routes_fra = df_all_routes_medium.query("(departure_airport_code == 'FRA' or arrival_airport_code == 'FRA')"
                                                  " and (departure_airport_code != 'LHR' and arrival_airport_code != 'LHR')"
                                                  " and (departure_airport_code != 'CDG' and arrival_airport_code != 'CDG')")

df_medium_routes_lhr = df_all_routes_medium.query("(departure_airport_code != 'FRA' and arrival_airport_code != 'FRA')"
                                                  " and (departure_airport_code != 'CDG' and arrival_airport_code != 'CDG')"
                                                  " and (departure_airport_code == 'LHR' or arrival_airport_code == 'LHR')"
                                                  ).iloc[np.r_[0, 2, 4, 10, 12, 14]]

df_medium_routes_cdg = df_all_routes_medium.query("(departure_airport_code != 'FRA' and arrival_airport_code != 'FRA')"
                                                  " and (departure_airport_code != 'LHR' and arrival_airport_code != 'LHR')"
                                                  " and (departure_airport_code == 'CDG' or arrival_airport_code == 'CDG')"
                                                  ).iloc[np.r_[1, 3, 5, 11, 13, 15]]

df_routes_medium = pd.concat([df_medium_routes_fra, df_medium_routes_lhr, df_medium_routes_cdg])

df_routes_medium["flights_per_day"] = 2

df_routes_medium.reset_index(drop=True, inplace=True)

df_routes_medium.insert(0, "line_number", 101 + df_routes_medium.index + len(df_routes_short))

df_routes_medium

,line_number,departure_airport_code,arrival_airport_code,distance_km,flights_per_day
0,191,FRA,DXB,4844,2
1,192,FRA,DOH,4588,2
2,193,FRA,CAI,2922,2
3,194,FRA,TLV,2954,2
4,195,FRA,BEY,2839,2
5,196,FRA,AMM,3045,2
6,197,FRA,ALG,1545,2
7,198,FRA,CMN,2277,2
8,199,FRA,TUN,1472,2
9,200,FRA,LCA,2637,2


In [19]:
df_routes_medium.flights_per_day.sum()

64

### Long-Haul Routes
#### Over 5000 km distance, all Routes connected to Primary Hub (Frankfurt)

In [26]:
df_all_routes_long = generate_all_possible_routes(airports_data_long)

df_routes_long = df_all_routes_long.copy().query("departure_airport_code == 'FRA' or arrival_airport_code == 'FRA'")

df_routes_long["flights_per_day"] = 1

df_routes_long.reset_index(drop=True, inplace=True)

df_routes_long.insert(0, "line_number", 101 + df_routes_long.index + len(df_routes_short) + len(df_routes_medium))

df_routes_long

,line_number,departure_airport_code,arrival_airport_code,distance_km,flights_per_day
0,223,FRA,JFK,6189,1
1,224,FRA,SFO,9148,1
2,225,FRA,YYZ,6343,1
3,226,FRA,HKG,9154,1
4,227,FRA,SIN,10278,1
5,228,FRA,ICN,8544,1
6,229,FRA,GRU,9798,1
7,230,FRA,NRT,9366,1
8,231,FRA,JNB,8690,1
9,232,FRA,SYD,16496,1


In [27]:
df_routes = (
    pd.concat([df_routes_short.copy(), df_routes_medium.copy(), df_routes_long.copy()])
    .sort_values("line_number").reset_index(drop=True)
)

random.seed(50)

route_price_cache = {}

def calculate_prices_pairwise_with_variation(dep, arr, distance_km_):
    # Order-independent key (same for FRA-SYD and SYD-FRA)
    route_key = tuple(sorted([dep, arr]))

    if route_key not in route_price_cache:
        # Base rates
        if distance_km_ < 1500:
            base_rate = random.uniform(0.10, 0.15)
        elif distance_km_ < 5000:
            base_rate = random.uniform(0.08, 0.12)
        else:
            base_rate = random.uniform(0.05, 0.10)

        # Base prices
        price_economy = distance_km_ * base_rate
        price_business = price_economy * random.uniform(2.3, 2.7)
        price_first = price_economy * random.uniform(4.8, 5.2)

        # Store base prices (no variation yet)
        route_price_cache[route_key] = (price_economy, price_business, price_first)

    # Apply small directional variation (±2%)
    base_prices = route_price_cache[route_key]
    directional_multiplier = random.uniform(0.98, 1.02)

    varied_prices = tuple(round(p * directional_multiplier, 2) for p in base_prices)

    return varied_prices

df_routes[["price_economy", "price_business", "price_first"]] = df_routes.apply(
    lambda row_: pd.Series(
        calculate_prices_pairwise_with_variation(
            row_["departure_airport_code"],
            row_["arrival_airport_code"],
            row_["distance_km"]
        )
    ),
    axis=1
)

df_routes

,line_number,departure_airport_code,arrival_airport_code,distance_km,flights_per_day,price_economy,price_business,price_first
0,101,FRA,LHR,654,2,80.83,194.51,408.58
1,102,FRA,BER,431,2,52.91,142.22,255.76
2,103,FRA,HAM,411,2,55.72,130.05,270.86
3,104,FRA,VIE,621,2,94.27,228.86,459.61
4,105,FRA,MUC,300,2,35.12,89.26,177.12
...,...,...,...,...,...,...,...,...
137,238,ICN,FRA,8544,1,676.10,1792.81,3380.44
138,239,GRU,FRA,9798,1,791.60,2064.55,4014.61
139,240,NRT,FRA,9366,1,734.11,1964.30,3717.74
140,241,JNB,FRA,8690,1,505.38,1258.51,2531.70


In [30]:
# Check for reversed route counterparts

# Step 1: Create a set of route tuples and reverse route tuples
routes = set(zip(df_routes['departure_airport_code'], df_routes['arrival_airport_code']))
reverse_routes = set((arr, dep) for dep, arr in routes)

# Step 2: Check if all routes have a reverse
missing_reverse_routes = routes - reverse_routes

# Print missing reverse routes if any
if missing_reverse_routes:
    print("Routes missing reverse counterparts:")
    for route in missing_reverse_routes:
        print(f"{route[0]} → {route[1]}")
else:
    print("✅ All routes have reverse counterparts.")

✅ All routes have reverse counterparts.


In [23]:
# total flights per day
df_routes.flights_per_day.sum()

264

In [12]:
if csv_settings == "imm":
    df_routes.to_csv(csv_folder_path + "routes.csv", index=False, sep=separator, decimal=dec)

### Initial Aircraft Locations

In [25]:
# needed for consistent flight scheduling

def assign_initial_locations(routes_df, aircraft_df):
    """
    Assigns initial aircraft locations based on departure airport frequencies.

    Parameters:
        routes_df (pd.DataFrame): Must contain 'departure_airport_code'.
        aircraft_df (pd.DataFrame): Must contain 'aircraft_id'.

    Returns:
        dict: {aircraft_id: initial_departure_airport_code}
    """
    departure_counts = routes_df["departure_airport_code"].value_counts().to_dict()
    num_aircraft = len(aircraft_df)
    total_routes = sum(departure_counts.values())

    # Step 1: Floor allocation and remainders
    allocations = {}
    remainders = {}

    for airport_, count in departure_counts.items():
        proportion = (count / total_routes) * num_aircraft
        allocations[airport_] = floor(proportion)
        remainders[airport_] = proportion - allocations[airport_]

    # Step 2: Distribute remaining aircraft by the largest remainder
    assigned_total = sum(allocations.values())
    remaining = num_aircraft - assigned_total
    for airport_ in sorted(remainders, key=remainders.get, reverse=True)[:remaining]:
        allocations[airport_] += 1

    # Step 3: Build the airport assignment list
    airport_list = []
    for airport_, count in allocations.items():
        airport_list.extend([airport_] * count)

    # Step 4: Shuffle and zip with aircraft IDs
    np.random.shuffle(airport_list)
    aircraft_ids = aircraft_df["aircraft_id"].tolist()

    return dict(zip(aircraft_ids, airport_list))

In [26]:
np.random.seed(50)
initial_locations_short_a = assign_initial_locations(df_routes_short, df_aircraft_short_a)
print(len(initial_locations_short_a))
initial_locations_short_a

17


{'AC1032': 'MUC',
 'AC1033': 'BRU',
 'AC1034': 'IST',
 'AC1030': 'FRA',
 'AC1016': 'FRA',
 'AC1017': 'LHR',
 'AC1015': 'VIE',
 'AC1011': 'VIE',
 'AC1013': 'HAM',
 'AC1026': 'BER',
 'AC1025': 'LHR',
 'AC1022': 'BER',
 'AC1021': 'FRA',
 'AC1004': 'MAD',
 'AC1006': 'CDG',
 'AC1007': 'FRA',
 'AC1002': 'ARN'}

In [27]:
np.random.seed(50)
initial_locations_short_b = assign_initial_locations(df_routes_short, df_aircraft_short_b)
print(len(initial_locations_short_b))
initial_locations_short_b

19


{'AC1029': 'BRU',
 'AC1031': 'LHR',
 'AC1035': 'VIE',
 'AC1028': 'MUC',
 'AC1010': 'FRA',
 'AC1014': 'MAD',
 'AC1012': 'LHR',
 'AC1018': 'FRA',
 'AC1009': 'ARN',
 'AC1027': 'VIE',
 'AC1024': 'HAM',
 'AC1020': 'BER',
 'AC1023': 'BER',
 'AC1019': 'FRA',
 'AC1001': 'FRA',
 'AC1005': 'CDG',
 'AC1003': 'HAM',
 'AC1008': 'FRA',
 'AC1000': 'IST'}

In [28]:
np.random.seed(50)
initial_locations_medium_a = assign_initial_locations(df_routes_medium, df_aircraft_medium_a)
print(len(initial_locations_medium_a))
initial_locations_medium_a

9


{'AC2004': 'CDG',
 'AC2002': 'DOH',
 'AC2001': 'CAI',
 'AC2008': 'FRA',
 'AC2011': 'FRA',
 'AC2010': 'DXB',
 'AC2015': 'LHR',
 'AC2018': 'TLV',
 'AC2017': 'FRA'}

In [29]:
np.random.seed(50)
initial_locations_medium_b = assign_initial_locations(df_routes_medium, df_aircraft_medium_b)
print(len(initial_locations_medium_b))
initial_locations_medium_b

11


{'AC2003': 'BEY',
 'AC2005': 'TLV',
 'AC2000': 'CAI',
 'AC2007': 'LHR',
 'AC2009': 'FRA',
 'AC2012': 'DXB',
 'AC2006': 'CDG',
 'AC2014': 'DOH',
 'AC2016': 'FRA',
 'AC2019': 'AMM',
 'AC2013': 'FRA'}

In [30]:
initial_locations_long = {}
for n in range(len(df_aircraft_long)):
    aircraft = df_aircraft_long.iloc[n]["aircraft_id"]
    initial_locations_long[aircraft] = "FRA"
initial_locations_long[df_aircraft_long["aircraft_id"][0]] = "SYD"
initial_locations_long[df_aircraft_long["aircraft_id"][1]] = "SIN"

print(len(initial_locations_long))
initial_locations_long

12


{'AC3000': 'SYD',
 'AC3001': 'SIN',
 'AC3002': 'FRA',
 'AC3003': 'FRA',
 'AC3004': 'FRA',
 'AC3005': 'FRA',
 'AC3006': 'FRA',
 'AC3007': 'FRA',
 'AC3008': 'FRA',
 'AC3009': 'FRA',
 'AC3010': 'FRA',
 'AC3011': 'FRA'}

In [31]:
initial_loc_short_med_a = {**initial_locations_short_a, **initial_locations_medium_a}
len(initial_loc_short_med_a)

26

In [32]:
initial_loc_short_med_b = {**initial_locations_short_b, **initial_locations_medium_b}
len(initial_loc_short_med_b)

30

## Prepare Weather Table

Hourly Weather data for each airport, subsequently used to generate Flight delays and cancellations

In [33]:
# Choose a start and end date for df_weather
start_date_weather = start_date_global
end_date_weather = end_date_global
sim_days_weather = sim_days_global

observation_hours = list(range(24))  # 0 to 23 hours for hourly weather

used_airport_codes = df_airports["airport_code"].tolist()

# Assign regions based on known climate zones
airport_regions = {
    "ALG": "Mediterranean", "AMM": "Desert", "AMS": "Temperate", "ARN": "Cold",
    "ATH": "Mediterranean", "BCN": "Mediterranean", "BER": "Temperate", "BEY": "Mediterranean",
    "BRU": "Temperate", "CAI": "Desert", "CDG": "Temperate", "CMN": "Mediterranean",
    "CPH": "Temperate", "DOH": "Desert", "DXB": "Desert", "FCO": "Mediterranean", "FRA": "Temperate",
    "GRU": "Subtropical", "HAM": "Temperate", "HEL": "Cold", "HKG": "Subtropical",
    "ICN": "Temperate", "IST": "Temperate", "JFK": "Temperate", "JNB": "Subtropical",
    "LCA": "Mediterranean", "LHR": "Temperate", "LIS": "Mediterranean", "MAD": "Mediterranean",
    "MUC": "Temperate", "NRT": "Temperate", "OSL": "Cold", "PMI": "Mediterranean",
    "SFO": "Mediterranean", "SIN": "Tropical", "SYD": "Subtropical", "TLV": "Mediterranean",
    "TUN": "Mediterranean", "VIE": "Temperate", "WAW": "Cold", "YYZ": "Cold", "ZRH": "Temperate"
}

# Temperature ranges by region and season
temperature_ranges_by_region_and_season = {
    "Tropical": {
        "Winter": (24, 32),
        "Spring": (25, 33),
        "Summer": (26, 34),
        "Autumn": (25, 33),
    },
    "Temperate": {
        "Winter": (-2, 8),
        "Spring": (8, 18),
        "Summer": (15, 28),
        "Autumn": (5, 15),
    },
    "Mediterranean": {
        "Winter": (5, 15),
        "Spring": (10, 22),
        "Summer": (20, 33),
        "Autumn": (12, 24),
    },
    "Cold": {
        "Winter": (-27, -5),
        "Spring": (-5, 5),
        "Summer": (5, 18),
        "Autumn": (-5, 5),
    },
    "Desert": {
        "Winter": (12, 22),
        "Spring": (20, 35),
        "Summer": (30, 46),
        "Autumn": (20, 34),
    },
    "Subtropical": {
        "Winter": (10, 20),
        "Spring": (16, 28),
        "Summer": (20, 34),
        "Autumn": (14, 26),
    }
}

# Weighted conditions by region and season
weather_conditions_by_region_and_season = {
    "Tropical": {
        "Winter": {"conditions": ["Clear", "Rain", "Thunderstorm", "Haze"], "weights": [0.35, 0.35, 0.13, 0.06]},
        "Spring": {"conditions": ["Clear", "Rain", "Thunderstorm", "Fog"], "weights": [0.3, 0.5, 0.13, 0.02]},
        "Summer": {"conditions": ["Rain", "Thunderstorm", "Haze", "Fog"], "weights": [0.6, 0.23, 0.1, 0.02]},
        "Autumn": {"conditions": ["Clear", "Rain", "Thunderstorm", "Fog"], "weights": [0.2, 0.45, 0.2, 0.05]},
    },
    "Temperate": {
        "Winter": {"conditions": ["Cloudy", "Snow", "Clear", "Rain", "Fog"], "weights": [0.325, 0.175, 0.25, 0.15, 0.04]},
        "Spring": {"conditions": ["Clear", "Cloudy", "Rain", "Thunderstorm", "Fog"], "weights": [0.25, 0.3, 0.3, 0.1, 0.03]},
        "Summer": {"conditions": ["Clear", "Cloudy", "Rain", "Thunderstorm", "Fog"], "weights": [0.4, 0.3, 0.2, 0.05, 0.02]},
        "Autumn": {"conditions": ["Cloudy", "Rain", "Clear", "Fog"], "weights": [0.275, 0.425, 0.15, 0.08]},
    },
    "Mediterranean": {
        "Winter": {"conditions": ["Clear", "Cloudy", "Rain", "Fog"], "weights": [0.2, 0.3, 0.45, 0.03]},
        "Spring": {"conditions": ["Clear", "Cloudy", "Rain", "Thunderstorm"], "weights": [0.3, 0.3, 0.3, 0.1]},
        "Summer": {"conditions": ["Clear", "Cloudy", "Thunderstorm", "Haze"], "weights": [0.6, 0.25, 0.1, 0.05]},
        "Autumn": {"conditions": ["Clear", "Cloudy", "Rain", "Thunderstorm"],"weights": [0.25, 0.3, 0.35, 0.1]}
    },
    "Cold": {
        "Winter": {"conditions": ["Snow", "Clear", "Cloudy", "Blizzard", "Fog"], "weights": [0.35, 0.25, 0.25, 0.1, 0.03]},
        "Spring": {"conditions": ["Snow", "Clear", "Cloudy", "Rain", "Fog"], "weights": [0.25, 0.25, 0.25, 0.2, 0.03]},
        "Summer": {"conditions": ["Clear", "Cloudy", "Rain", "Fog"], "weights": [0.32, 0.315, 0.315, 0.03]},
        "Autumn": {"conditions": ["Snow", "Cloudy", "Clear", "Fog"], "weights": [0.315, 0.32, 0.315, 0.04]},
    },
    "Desert": {
        "Winter": {"conditions": ["Clear", "Haze"], "weights": [0.8, 0.2]},
        "Spring": {"conditions": ["Clear", "Haze", "Dust", "Sandstorm"], "weights": [0.7, 0.25, 0.07, 0.07]},
        "Summer": {"conditions": ["Clear", "Haze", "Dust", "Sandstorm"], "weights": [0.8, 0.15, 0.07, 0.07]},
        "Autumn": {"conditions": ["Clear", "Haze", "Dust"], "weights": [0.725, 0.2, 0.075]},
    },
    "Subtropical": {
        "Winter": {"conditions": ["Clear", "Cloudy", "Rain", "Fog"], "weights": [0.315, 0.32, 0.315, 0.03]},
        "Spring": {"conditions": ["Clear", "Cloudy", "Rain", "Thunderstorm"], "weights": [0.2, 0.3, 0.35, 0.13]},
        "Summer": {"conditions": ["Rain", "Clear", "Thunderstorm", "Fog"], "weights": [0.4, 0.4, 0.15, 0.03]},
        "Autumn": {"conditions": ["Clear", "Cloudy", "Rain", "Fog"], "weights": [0.315, 0.315, 0.32, 0.04]},
    }
}

random.seed(42)

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

def visibility_by_condition(condition_: str) -> float:
    condition_ = condition_.lower()
    if condition_ in ["fog", "blizzard", "sandstorm", "dust"]:
        return round(random.uniform(0.1, 1.7), 1)
    elif condition_ in ["thunderstorm", "snow", "haze"]:
        return round(random.uniform(1.7, 5), 1)
    elif condition_ in ["rain", "cloudy"]:
        return round(random.uniform(5, 15), 1)
    else:  # clear or other mild conditions
        return round(random.uniform(20, 50), 1)

def wind_speed_by_condition(condition_: str) -> float:
    condition_ = condition_.lower()
    if condition_ in ["sandstorm", "dust"]:
        return round(random.uniform(60, 100), 1)
    elif condition_ in ["thunderstorm", "blizzard"]:
        return round(random.uniform(40, 70), 1)
    elif condition_ in ["rain", "snow"]:
        return round(random.uniform(15, 30), 1)
    elif condition_ in ["cloudy", "haze"]:
        return round(random.uniform(5, 15), 1)
    elif condition_ in ["fog", "clear"]:
        return round(random.uniform(0, 5), 1)
    else:
        return round(random.uniform(5, 20), 1)  # fallback for any new condition

weather_data = []
observation_id = 1

for airport_code in used_airport_codes:
    region = airport_regions.get(airport_code, "Temperate")

    current_date = start_date_weather
    while current_date < end_date_weather:
        for hour in observation_hours:
            obs_time = current_date.replace(hour=hour, minute=0, second=0)
            season = get_season(current_date.month)
            temp_min, temp_max = temperature_ranges_by_region_and_season[region][season]
            temp = round(random.uniform(temp_min, temp_max), 1)
            conditions = weather_conditions_by_region_and_season[region][season]["conditions"]
            condition = random.choices(conditions, weather_conditions_by_region_and_season[region][season]["weights"], k=1)[0]
            visibility = visibility_by_condition(condition) # km
            wind_speed = wind_speed_by_condition(condition) # km/h

            weather_data.append({
                "weather_id": observation_id,
                "airport_code": airport_code,
                "observation_time": obs_time,
                "season": season,
                "temperature_celsius": temp,
                "wind_speed_kmh": wind_speed,
                "visibility_km": visibility,
                "condition_description": condition
            })
            observation_id += 1

        current_date += timedelta(days=1)

df_weather = pd.DataFrame(weather_data)

df_weather

,weather_id,airport_code,observation_time,season,temperature_celsius,wind_speed_kmh,visibility_km,condition_description
0,1,ALG,2022-01-01 00:00:00,Winter,11.4,1.1,28.3,Clear
1,2,ALG,2022-01-01 01:00:00,Winter,12.4,16.3,13.9,Rain
2,3,ALG,2022-01-01 02:00:00,Winter,9.2,2.5,26.6,Clear
3,4,ALG,2022-01-01 03:00:00,Winter,5.3,2.7,39.5,Clear
4,5,ALG,2022-01-01 04:00:00,Winter,7.2,15.1,13.1,Rain
...,...,...,...,...,...,...,...,...
1104763,1104764,ZRH,2024-12-31 19:00:00,Winter,-0.9,10.8,14.1,Cloudy
1104764,1104765,ZRH,2024-12-31 20:00:00,Winter,-1.2,4.6,48.4,Clear
1104765,1104766,ZRH,2024-12-31 21:00:00,Winter,2.3,5.3,8.4,Cloudy
1104766,1104767,ZRH,2024-12-31 22:00:00,Winter,5.6,9.3,8.7,Cloudy


In [34]:
# hazardous condition counts

t1 = df_weather[df_weather["visibility_km"] < 1.0]["visibility_km"].count()
t2 = df_weather[df_weather["wind_speed_kmh"] > 70.0]["wind_speed_kmh"].count()
t3 = df_weather[(df_weather["temperature_celsius"] < -25.0) | (df_weather["temperature_celsius"] > 45.0)]["temperature_celsius"].count()
t4 = df_weather[df_weather["condition_description"].isin(["Fog"])]["condition_description"].count()
t5 = df_weather[df_weather["condition_description"].isin(["Blizzard"])]["condition_description"].count()
t6 = df_weather[df_weather["condition_description"].isin(["Sandstorm"])]["condition_description"].count()

print("visability < 1.0:", t1)
print("wind speed > 70.0:", t2)
print("temperature < -25.0 or > 45.0:", t3)
print("fog:", t4)
print("blizzard:", t5)
print("sandstorm:", t6)
print("\nsum:", sum([t2, t3, t4, t5, t5, t6]))

visability < 1.0: 21196
wind speed > 70.0: 6697
temperature < -25.0 or > 45.0: 4317
fog: 27831
blizzard: 3367

sum: 45579


## Prepare Flights Table

Core table of the database, challenges:
- somewhat realistic occupancy with variation by travel season, weekdays, holidays, and randomized route demands
    (function: get_passenger_multiplier and route_demand_map (included in the main function, create_flights)
- the weather table should determine flight delays and cancellations due to weather
    (functions: check_weather_cancellation, calculate_weather_delay)
- somewhat realistic turnaround times, varied by aircraft size and flight distance (function: calculate_turnaround_minutes)
- key challenge: consistent flight scheduling
    - flights should be scheduled based on aircraft availability; starting from it's initial location
      the create_flights function must keep track of each aircraft's location at any given point in time
    - short- and medium-haul routes are supposed to have two flights per day;
      to avoid having both flights belonging to a certain route being scheduled too close to each other
      the two daily flights for short- and medium-haul routes will be generated by two separate function calls (a and b),
      with the second one having an offset of 6.5 hours, long-haul flights require only one function call for one flight per day;
      flights_short_medium_a, flights_short_medium_b, and flights_long will be combined as df_flights
- the flights table will be finalized after updating passenger counts for missed check-ins, which will require generating
  the bookings table first (this means the passenger counts that will be calculated with create_flights are really booking counts,
  not the actual numbers of people that boarded a plane)

In [35]:
def get_passenger_multiplier(flight_date_: datetime, line_number_: int, distance_: float, random_seed, route_demand_map) -> float:
    weekday = flight_date_.weekday()
    month = flight_date_.month
    date_only = flight_date_.date()

    # Holidays: Easter, Winter, Summer
    holiday_ranges = [
        pd.date_range("2022-04-15", "2022-04-18"), pd.date_range("2023-04-07", "2023-04-10"), pd.date_range("2024-03-29", "2024-04-02"),
        pd.date_range("2022-12-20", "2022-12-31"), pd.date_range("2023-12-20", "2023-12-31"), pd.date_range("2024-12-20", "2024-12-31"),
        pd.date_range("2022-07-01", "2022-07-08"), pd.date_range("2023-07-01", "2023-07-08"), pd.date_range("2024-07-01", "2024-07-08"),
    ]
    holiday_boost = any(date_only in rng for rng in holiday_ranges)

    # Weekday modifier
    if weekday in [4, 6]:  # Friday, Sunday
        weekday_mult = 1.1
    elif weekday in [1, 5]:  # Tuesday, Saturday
        weekday_mult = 0.9
    else:
        weekday_mult = 1.0

    # Seasonal variation
    if month in [6, 7, 8, 12]:  # Summer + December
        seasonal_mult = 1.1
    elif month in [1, 2, 11]:   # Off-season
        seasonal_mult = 0.9
    else:
        seasonal_mult = 1.0

    # Route-level demand modifier
    route_mult = route_demand_map.get(line_number_, 1.0)

    # Distance modifier
    if distance_ < 1500:
        distance_mult = 0.95  # Short-haul
    elif distance_ < 5000:
        distance_mult = 1.0   # Medium-haul
    else:
        distance_mult = 1.1   # Long-haul

    # Holiday boost overrides all
    if holiday_boost:
        return round(random_seed.uniform(1.25, 1.35) * route_mult, 2)

    # Add minor random variation on regular days
    random_spike = random_seed.uniform(0.95, 1.05)

    # Combine all multipliers
    base_mult = weekday_mult * seasonal_mult * route_mult * distance_mult * random_spike

    return round(base_mult, 2)

In [36]:
def check_weather_cancellation(scheduled_departure, dep_airport_code, distance_km_, weather_df):
    # Time window to check
    hours_to_check = 4 if distance_km_ > 5000 else 3 if distance_km_ > 1500 else 2
    times_to_check = [scheduled_departure + timedelta(hours=h) for h in range(hours_to_check)]

    cancel_reasons = []
    for observ_time in times_to_check:
        key = (dep_airport_code, observ_time.replace(minute=0, second=0, microsecond=0))
        if key not in weather_df.index:
            continue

        weather = weather_df.loc[key]

        if weather["visibility_km"] < 1.0:
            cancel_reasons.append("Low visibility")
        if weather["wind_speed_kmh"] > 70.0:
            cancel_reasons.append("Strong winds")
        if weather["temperature_celsius"] < -25.0 or weather["temperature_celsius"] > 45.0:
            cancel_reasons.append("Extreme temperature")
        if weather["condition_description"] in {"Fog", "Blizzard", "Sandstorm"}:
            cancel_reasons.append(weather["condition_description"])

    # Return True if bad conditions persisted for the entire window
    if len(cancel_reasons) == hours_to_check:
        return True
    return False

In [37]:
def calculate_weather_delay(
    scheduled_time,
    airport_code_,
    distance_km_,
    weather_df,
    random_seed,
    phase="departure"  # or "arrival"
):
    """
    Calculates weather-related delay in minutes based on scheduled time and airport.
    Returns delay in minutes (0 if no delay due to weather).
    """
    assert phase in {"departure", "arrival"}, "phase must be 'departure' or 'arrival'"

    # Determine the time window and direction of checking
    if phase == "departure":
        hours_to_check = 4 if distance_km_ > 5000 else 3 if distance_km_ > 1500 else 2
        times_to_check = [scheduled_time + timedelta(hours=h) for h in range(hours_to_check)]
    else:  # arrival
        hours_to_check = 2 if distance_km_ > 1500 else 1
        times_to_check = [scheduled_time - timedelta(hours=h) for h in range(hours_to_check)]

    delay_minutes = 0

    for time_to_check in times_to_check:
        key = (airport_code_, time_to_check.replace(minute=0, second=0, microsecond=0))
        if key not in weather_df.index:
            continue

        weather = weather_df.loc[key]
        condition_ = weather["condition_description"].lower()
        vis = weather["visibility_km"]
        wind = weather["wind_speed_kmh"]
        temp_ = weather["temperature_celsius"]

        # Accumulate delay for impactful conditions
        if vis < 1.0:
            delay_minutes += random_seed.randint(15, 45) if phase == "departure" else random_seed.randint(10, 30)
        if wind > 70.0:
            delay_minutes += random_seed.randint(30, 90) if phase == "departure" else random_seed.randint(15, 45)
        if condition_ in {"fog", "blizzard", "sandstorm"}:
            delay_minutes += random_seed.randint(30, 90) if phase == "departure" else random_seed.randint(20, 60)
        if temp_ < -25.0 or temp_ > 45.0:
            if phase == "departure":
                delay_minutes += random_seed.randint(15, 45)  # less relevant for arrival

    return delay_minutes

In [38]:
def calculate_turnaround_minutes(seat_capacity_, distance_km_, random_seed):
    base = 40

    # Seat capacity impact (larger aircraft take longer)
    if seat_capacity_ >= 300:
        base += 20
    elif seat_capacity_ >= 220:
        base += 15
    elif seat_capacity_ >= 170:
        base += 10
    elif seat_capacity_ >= 120:
        base += 5
    # else: base += 0 for smaller aircraft

    # Distance impact (longer flights need more time on the ground)
    if distance_km_ > 8000:
        base += 10
    elif distance_km_ > 3000:
        base += 5
    elif distance_km_ < 800:
        base -= 5

    # Light randomization for variation
    base += random_seed.randint(-3, 6)

    # Enforce a realistic floor
    return max(base, 30)

In [39]:
# the number of days to simulate and start date for df_flights should match weather data (a shorter range of days can be chosen, not a longer one)
def create_flights(departure_offset, random_other, aircraft_, initial_locations, routes_,
                   short_dummies, medium_dummies, sim_days_=sim_days_weather):

    start_date = start_date_global

    # Seeds
    rand_main = random.Random(50)
    np_seed = np.random.default_rng(50)
    rand_other = random_other

    # Weather indexing for speed
    df_weather_multi_index = df_weather.copy()
    df_weather_multi_index.set_index(["airport_code", "observation_time"], inplace=True)

    # Active aircraft
    active_aircraft = aircraft_.copy()

    aircraft_status = {}
    for ac_id, ap in initial_locations.items():
        aircraft_status[ac_id] = {
            "available_from": start_date,
            "location": ap,
            "flight_attempts": 0,
            "grounded": False
        }

    # Route demand tiers
    route_numbers = routes_["line_number"].tolist()
    rand_main.shuffle(route_numbers)

    n_ = len(route_numbers)
    route_demand_map = {}
    for i, route_ in enumerate(route_numbers):
        if i < n_ * 0.15:
            route_demand_map[route_] = 1.15
        elif i < n_ * 0.4:
            route_demand_map[route_] = 1.1
        elif i < n_ * 0.9:
            route_demand_map[route_] = 1.0
        else:
            route_demand_map[route_] = 0.95

    # Flight and delay params
    delay_reasons_dep = ["Technical issue", "Crew delay", "Air traffic control"]
    delay_weights_dep = [0.5, 0.3, 0.2]
    delay_reasons_arr = ["Airspace congestion", "Ground handling"]
    delay_weights_arr = [0.6, 0.4]

    cancellation_reasons = ["Technical failure", "Crew unavailable", "Airport closure"]
    cancellation_weights = [0.6, 0.3, 0.1]

    # Simulation
    flight_records = []
    flight_record_keys = ["flight_date", "line_number", "aircraft_id", "bookings_total", "scheduled_departure",
                          "scheduled_arrival", "actual_departure", "actual_arrival", "cancelled", "cancellation_reason",
                          "delay_reason_dep", "delay_reason_arr", "seat_capacity", "distance_km"]


    for day_offset in range(sim_days_):
        flight_date_ = start_date + timedelta(days=day_offset)
        day_end = datetime.combine(flight_date_.date(), datetime.max.time())
        active_this_day = set()
        route_usage_today = defaultdict(int) # key: line_number
        max_flights_per_day = 8 # adjustable cap for realism and performance
        total_aircraft_grounded = 0

        # Reset the flight attempts per day
        for ac_id in aircraft_status:
            aircraft_status[ac_id]["flight_attempts"] = 0
            aircraft_status[ac_id]["grounded"] = False

        while total_aircraft_grounded < len(initial_locations) + 1:
            if total_aircraft_grounded == len(initial_locations):
                for line_number in route_numbers:
                    if route_usage_today[line_number] < 1:
                        rnd_seconds = rand_other.randint(0, int((day_end - flight_date_).total_seconds()))
                        scheduled_dep = flight_date_ + timedelta(seconds=rnd_seconds) + departure_offset ##############################################
                        distance_km = routes_[routes_["line_number"] == line_number]["distance_km"].values[0]
                        duration = timedelta(hours=distance_km / 900 + rand_other.uniform(0.1, 0.3))
                        scheduled_arr = scheduled_dep + duration

                        if line_number in df_routes_short["line_number"].tolist():
                            dummy_aircraft = short_dummies
                        elif line_number in df_routes_medium["line_number"].tolist():
                            dummy_aircraft = medium_dummies
                        else:
                            dummy_aircraft = df_aircraft_long["aircraft_id"].tolist()

                        ac_id = rand_other.choice(dummy_aircraft)

                        seat_capacity = aircraft_[aircraft_["aircraft_id"] == ac_id]["seat_capacity"].values[0]

                        # Set a minimum passenger threshold
                        min_passengers = seat_capacity * 0.5

                        # Calculate Passenger count
                        multiplier = get_passenger_multiplier(flight_date_, line_number, distance_=distance_km,
                                                              random_seed=rand_other, route_demand_map=route_demand_map)
                        base_passengers = seat_capacity * 0.75
                        passengers = min(int(max(base_passengers * multiplier, min_passengers)), int(seat_capacity * 0.99))

                        flight_records.append({
                        flight_record_keys[0]: flight_date_.date(),
                        flight_record_keys[1]: line_number,
                        flight_record_keys[2]: ac_id,
                        flight_record_keys[3]: passengers,
                        flight_record_keys[4]: scheduled_dep,
                        flight_record_keys[5]: scheduled_arr,
                        flight_record_keys[6]: pd.NaT,
                        flight_record_keys[7]: pd.NaT,
                        flight_record_keys[8]: True,
                        flight_record_keys[9]: "Aircraft unavailable",
                        flight_record_keys[10]: None,
                        flight_record_keys[11]: None,
                        flight_record_keys[12]: seat_capacity,
                        flight_record_keys[13]: distance_km
                    })

                        route_usage_today[line_number] += 1 ###

                total_aircraft_grounded += 1

            for ac_id, status in aircraft_status.items():

                if status["grounded"]:
                    continue

                if status["available_from"] > day_end:
                    status["grounded"] = True
                    total_aircraft_grounded += 1
                    continue

                if status["flight_attempts"] == max_flights_per_day:
                    status["grounded"] = True
                    total_aircraft_grounded += 1
                    continue

                status["available_from"] = max(status["available_from"], datetime.combine(flight_date_.date(), datetime.min.time()) + departure_offset)  ###################
                location = status["location"]
                scheduled = False

                aircraft_row = active_aircraft[active_aircraft["aircraft_id"] == ac_id]

                if aircraft_row.empty:
                    print(f"AIRCRAFT {ac_id} IS NOT AT {location}!")
                    status["grounded"] = True
                    total_aircraft_grounded += 1
                    continue

                aircraft_range = aircraft_row["range_km"].values[0]
                seat_capacity = aircraft_row["seat_capacity"].values[0]

                flag = False

                if aircraft_range >= df_aircraft_long["range_km"].min():
                    eligible_routes = df_routes_long[df_routes_long["departure_airport_code"] == location]
                    flag = True

                elif aircraft_range >= df_aircraft_medium["range_km"].min() and location in df_routes_medium["departure_airport_code"].tolist():
                    for ln in df_routes_medium["line_number"].tolist():
                        if route_usage_today[ln] == 0:
                            eligible_routes = df_routes_medium[df_routes_medium["departure_airport_code"] == location]
                            flag = True
                            break
                        else:
                            continue

                if not flag:
                    eligible_routes = routes_[routes_["departure_airport_code"] == location]
                    eligible_routes = eligible_routes[eligible_routes["distance_km"] <= aircraft_range]

                # noinspection PyUnboundLocalVariable
                if eligible_routes.empty:
                    # Advance time so we don't get stuck here forever
                    print(f"AIRCRAFT {ac_id} IS STUCK IN {location}!")
                    status["grounded"] = True
                    total_aircraft_grounded += 1
                    continue


                for _, route_ in eligible_routes.iterrows():
                    line_number = route_["line_number"]
                    if route_usage_today[line_number] == 1:
                        continue  # Skip this route — it's already full
                    distance_km = route_["distance_km"]
                    dest_airport = route_["arrival_airport_code"]

                    duration = timedelta(hours=distance_km / 900 + rand_other.uniform(0.1, 0.3))
                    scheduled_dep = status["available_from"]
                    scheduled_arr = scheduled_dep + duration

                    # Set a minimum passenger threshold
                    min_passengers = seat_capacity * 0.5

                    # Calculate Passenger count
                    multiplier = get_passenger_multiplier(flight_date_, line_number, distance_=distance_km,
                                                          random_seed=rand_other, route_demand_map=route_demand_map)
                    base_passengers = seat_capacity * 0.75
                    passengers = min(int(max(base_passengers * multiplier, min_passengers)), int(seat_capacity * 0.99))

                    weather_cancellation = check_weather_cancellation(
                        scheduled_departure=scheduled_dep,
                        dep_airport_code=location,
                        distance_km_=distance_km,
                        weather_df=df_weather_multi_index
                    )

                    if weather_cancellation:
                        is_cancelled = True
                        route_usage_today[line_number] += 1
                        cancellation_reason = "Weather"
                        actual_dep = actual_arr = None
                        delay_reason_dep = delay_reason_arr = None
                        status["location"] = dest_airport
                        status["available_from"] = scheduled_arr + timedelta(minutes=90)
                    else:
                        is_cancelled = rand_main.random() < 0.017
                        cancellation_reason = rand_main.choices(cancellation_reasons, cancellation_weights)[0] if is_cancelled else None

                        if is_cancelled:
                            route_usage_today[line_number] += 1
                            actual_dep = actual_arr = None
                            delay_reason_dep = delay_reason_arr = None
                            status["location"] = dest_airport
                            status["available_from"] = scheduled_arr + timedelta(minutes=90)
                        else:
                            delay_dep_min = calculate_weather_delay(
                                scheduled_time=scheduled_dep,
                                airport_code_=location,
                                distance_km_=distance_km,
                                weather_df=df_weather_multi_index,
                                random_seed=rand_other,
                                phase="departure"
                            )
                            if delay_dep_min <= 15:
                                delay_dep_min = max(0, int(np_seed.exponential(scale=9)))
                                delay_reason_dep = rand_other.choices(delay_reasons_dep, delay_weights_dep)[0] if delay_dep_min > 15 else None
                            else:
                                delay_reason_dep = "Weather"

                            scheduled_arr_with_dep_delay = scheduled_arr + timedelta(minutes=delay_dep_min)
                            delay_arr_min = calculate_weather_delay(
                                scheduled_time=scheduled_arr_with_dep_delay,
                                airport_code_=dest_airport,
                                distance_km_=distance_km,
                                weather_df=df_weather_multi_index,
                                random_seed=rand_other,
                                phase="arrival"
                            )
                            if delay_arr_min <= 15:
                                delta = rand_other.randint(-5, 10)
                                delay_arr_min = max(0, delay_dep_min + delta)

                                if delay_dep_min > 15:
                                    delay_reason_arr = "Late departure" if delay_arr_min > 15 else None
                                else:
                                    delay_reason_arr = rand_other.choices(delay_reasons_arr, delay_weights_arr)[0] if delay_arr_min > 15 else None
                            else:
                                delay_reason_arr = "Weather"
                                delay_arr_min = delay_dep_min + delay_arr_min

                            actual_dep = scheduled_dep + timedelta(minutes=delay_dep_min)
                            actual_arr = scheduled_arr + timedelta(minutes=delay_arr_min)

                            turnaround = calculate_turnaround_minutes(seat_capacity, distance_km, rand_other)
                            status["available_from"] = actual_arr + timedelta(minutes=turnaround)
                            status["location"] = dest_airport
                            route_usage_today[line_number] += 1
                            status["flight_attempts"] += 1

                    flight_records.append({
                        flight_record_keys[0]: flight_date_.date(),
                        flight_record_keys[1]: line_number,
                        flight_record_keys[2]: ac_id,
                        flight_record_keys[3]: passengers,
                        flight_record_keys[4]: scheduled_dep,
                        flight_record_keys[5]: scheduled_arr,
                        flight_record_keys[6]: actual_dep,
                        flight_record_keys[7]: actual_arr,
                        flight_record_keys[8]: is_cancelled,
                        flight_record_keys[9]: cancellation_reason,
                        flight_record_keys[10]: delay_reason_dep,
                        flight_record_keys[11]: delay_reason_arr,
                        flight_record_keys[12]: seat_capacity,
                        flight_record_keys[13]: distance_km
                    })

                    active_this_day.add(ac_id)
                    scheduled = True
                    break  # Try next available route in next loop

                if not scheduled:
                    status["grounded"] = True
                    total_aircraft_grounded += 1

    # Create DataFrame
    df_flights_ = pd.DataFrame(flight_records)
    df_flights_["flight_date"] = pd.to_datetime(df_flights_["flight_date"])

    return df_flights_

In [40]:
sim_days = sim_days_global

In [41]:
departure_offset_a = timedelta(hours=1)
rand_other_a = random.Random(50)
aircraft_short_medium_a = (
    pd.concat([df_aircraft_short_a.copy(), df_aircraft_medium_a.copy()]).sort_values("aircraft_id").reset_index(drop=True))
initial_locations_a = initial_loc_short_med_a
routes_short_medium = (
    pd.concat([df_routes_short.copy(), df_routes_medium.copy()]).sort_values("line_number").reset_index(drop=True))

short_dummies_a = df_aircraft_short_a["aircraft_id"].tolist()
medium_dummies_a = df_aircraft_medium_a["aircraft_id"].tolist()

flights_short_medium_a = create_flights(departure_offset_a, rand_other_a, aircraft_short_medium_a,
                                        initial_locations_a, routes_short_medium, short_dummies_a,
                                        medium_dummies_a, sim_days_=sim_days)

flights_short_medium_a

,flight_date,line_number,aircraft_id,bookings_total,scheduled_departure,scheduled_arrival,actual_departure,actual_arrival,cancelled,cancellation_reason,delay_reason_dep,delay_reason_arr,seat_capacity,distance_km
0,2022-01-01,154,AC1032,90,2022-01-01 01:00:00.000000,2022-01-01 01:31:58.226330,2022-01-01 01:13:00.000000,2022-01-01 01:46:58.226330,False,None,None,None,145,300
1,2022-01-01,186,AC1033,94,2022-01-01 01:00:00.000000,2022-01-01 01:35:28.169815,2022-01-01 01:04:00.000000,2022-01-01 01:44:28.169815,False,None,None,None,145,304
2,2022-01-01,164,AC1034,80,2022-01-01 01:00:00.000000,2022-01-01 03:16:39.039647,2022-01-01 01:15:00.000000,2022-01-01 03:30:39.039647,False,None,None,None,145,1838
3,2022-01-01,101,AC1030,88,2022-01-01 01:00:00.000000,2022-01-01 01:59:30.197172,2022-01-01 01:07:00.000000,2022-01-01 02:11:30.197172,False,None,None,None,145,654
4,2022-01-01,102,AC1016,125,2022-01-01 01:00:00.000000,2022-01-01 01:37:00.057670,NaT,NaT,True,Technical failure,None,None,180,431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133707,2024-12-31,172,AC1022,95,2024-12-31 11:38:40.307542,2024-12-31 12:42:05.160523,2024-12-31 11:51:40.307542,2024-12-31 12:52:05.160523,False,None,None,None,132,728
133708,2024-12-31,181,AC1032,106,2024-12-31 17:03:07.220073,2024-12-31 18:09:48.851471,2024-12-31 17:17:07.220073,2024-12-31 18:27:48.851471,False,None,None,Airspace congestion,145,855
133709,2024-12-31,189,AC1034,97,2024-12-31 15:49:24.499952,2024-12-31 17:29:16.261367,2024-12-31 15:49:24.499952,2024-12-31 17:32:16.261367,False,None,None,None,145,1316
133710,2024-12-31,190,AC1030,109,2024-12-31 11:56:01.397928,2024-12-31 12:19:56.401302,2024-12-31 12:11:01.397928,2024-12-31 12:42:56.401302,False,None,None,Airspace congestion,145,252


In [42]:
departure_offset_b = timedelta(hours=6.5)
rand_other_b = random.Random(69)
aircraft_short_medium_b = (
    pd.concat([df_aircraft_short_b.copy(), df_aircraft_medium_b.copy()]).sort_values("aircraft_id").reset_index(drop=True))
initial_locations_b = initial_loc_short_med_b

short_dummies_b = df_aircraft_short_b["aircraft_id"].tolist()
medium_dummies_b = df_aircraft_medium_b["aircraft_id"].tolist()

flights_short_medium_b = create_flights(departure_offset_b, rand_other_b, aircraft_short_medium_b,
                                        initial_locations_b, routes_short_medium, short_dummies_b,
                                        medium_dummies_b, sim_days_=sim_days)

flights_short_medium_b

,flight_date,line_number,aircraft_id,bookings_total,scheduled_departure,scheduled_arrival,actual_departure,actual_arrival,cancelled,cancellation_reason,delay_reason_dep,delay_reason_arr,seat_capacity,distance_km
0,2022-01-01,186,AC1029,92,2022-01-01 06:30:00.000000,2022-01-01 07:04:28.653486,2022-01-01 06:43:00.000000,2022-01-01 07:17:28.653486,False,None,None,None,145,304
1,2022-01-01,122,AC1031,90,2022-01-01 06:30:00.000000,2022-01-01 07:26:51.689178,2022-01-01 06:34:00.000000,2022-01-01 07:38:51.689178,False,None,None,None,145,654
2,2022-01-01,146,AC1035,96,2022-01-01 06:30:00.000000,2022-01-01 07:27:20.766397,2022-01-01 06:45:00.000000,2022-01-01 07:47:20.766397,False,None,None,Airspace congestion,145,621
3,2022-01-01,154,AC1028,96,2022-01-01 06:30:00.000000,2022-01-01 06:57:42.208280,2022-01-01 06:37:00.000000,2022-01-01 07:01:42.208280,False,None,None,None,145,300
4,2022-01-01,101,AC1010,101,2022-01-01 06:30:00.000000,2022-01-01 07:23:02.024246,NaT,NaT,True,Technical failure,None,None,180,654
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133707,2024-12-31,185,AC1001,156,2024-12-31 14:40:02.792172,2024-12-31 16:49:59.466832,2024-12-31 15:54:02.792172,2024-12-31 17:59:59.466832,False,None,Weather,Late departure,211,1817
133708,2024-12-31,188,AC1005,170,2024-12-31 14:04:39.763471,2024-12-31 15:19:33.569944,2024-12-31 14:04:39.763471,2024-12-31 15:27:33.569944,False,None,None,None,211,925
133709,2024-12-31,210,AC1008,151,2024-12-31 18:13:48.890228,2024-12-31 21:22:43.598154,2024-12-31 18:14:48.890228,2024-12-31 21:22:43.598154,False,None,None,None,211,2637
133710,2024-12-31,189,AC1035,101,2024-12-31 20:27:02.760760,2024-12-31 22:07:28.215977,2024-12-31 20:52:02.760760,2024-12-31 22:30:28.215977,False,None,Technical issue,Late departure,145,1316


In [43]:
departure_offset_long = timedelta(hours=0)
rand_other_long = random.Random(50)
aircraft_long = df_aircraft_long
#initial_locations_long = initial_locations_long
routes_long = df_routes_long

short_dummies_all = df_aircraft_short["aircraft_id"].tolist()
medium_dummies_all = df_aircraft_medium["aircraft_id"].tolist()

flights_long = create_flights(departure_offset_long, rand_other_long, aircraft_long,
                              initial_locations_long, routes_long, short_dummies_all,
                              medium_dummies_all, sim_days_=sim_days)

flights_long

,flight_date,line_number,aircraft_id,bookings_total,scheduled_departure,scheduled_arrival,actual_departure,actual_arrival,cancelled,cancellation_reason,delay_reason_dep,delay_reason_arr,seat_capacity,distance_km
0,2022-01-01,242,AC3000,176,2022-01-01 00:00:00.000000,2022-01-01 18:31:42.226330,2022-01-01 00:13:00.000000,2022-01-01 18:46:42.226330,False,None,None,None,270,16496
1,2022-01-01,237,AC3001,194,2022-01-01 00:00:00.000000,2022-01-01 11:40:24.169815,2022-01-01 00:04:00.000000,2022-01-01 11:49:24.169815,False,None,None,None,270,10278
2,2022-01-01,223,AC3002,190,2022-01-01 00:00:00.000000,2022-01-01 07:06:43.039647,2022-01-01 00:15:00.000000,2022-01-01 07:20:43.039647,False,None,None,None,270,6189
3,2022-01-01,224,AC3003,188,2022-01-01 00:00:00.000000,2022-01-01 10:25:46.197172,2022-01-01 00:07:00.000000,2022-01-01 11:26:46.197172,False,None,None,Weather,270,9148
4,2022-01-01,225,AC3004,206,2022-01-01 00:00:00.000000,2022-01-01 07:09:41.491799,2022-01-01 00:02:00.000000,2022-01-01 08:09:41.491799,False,None,None,Weather,270,6343
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21915,2024-12-31,229,AC3003,240,2024-12-31 23:52:44.012716,2025-01-01 11:00:58.073750,2024-12-31 23:58:44.012716,2025-01-01 11:05:58.073750,False,None,None,None,270,9798
21916,2024-12-31,230,AC3004,230,2024-12-31 13:04:49.287791,2024-12-31 23:41:50.143004,2024-12-31 13:15:49.287791,2024-12-31 23:54:50.143004,False,None,None,None,270,9366
21917,2024-12-31,231,AC3005,256,2024-12-31 11:13:02.154258,2024-12-31 21:03:49.757605,2024-12-31 11:39:02.154258,2024-12-31 21:30:49.757605,False,None,Technical issue,Late departure,317,8690
21918,2024-12-31,232,AC3009,294,2024-12-31 11:47:44.764324,2025-01-01 06:24:49.942215,2024-12-31 11:48:44.764324,2025-01-01 06:26:49.942215,False,None,None,None,317,16496


#### Investigate Results

In [44]:
flights_short_medium_a.query("cancellation_reason == 'Aircraft unavailable'")

,flight_date,line_number,aircraft_id,bookings_total,scheduled_departure,scheduled_arrival,actual_departure,actual_arrival,cancelled,cancellation_reason,delay_reason_dep,delay_reason_arr,seat_capacity,distance_km
12565,2022-04-13,209,AC2011,141,2022-04-13 08:06:31,2022-04-13 09:56:10.582106,NaT,NaT,True,Aircraft unavailable,None,None,193,1472
13053,2022-04-17,216,AC2008,166,2022-04-17 05:32:00,2022-04-17 09:33:58.860030,NaT,NaT,True,Aircraft unavailable,None,None,193,3481
13419,2022-04-20,222,AC2001,138,2022-04-20 05:54:44,2022-04-20 09:49:02.247140,NaT,NaT,True,Aircraft unavailable,None,None,189,3381
14883,2022-05-02,210,AC2011,137,2022-05-02 23:41:31,2022-05-03 02:49:27.349081,NaT,NaT,True,Aircraft unavailable,None,None,193,2637


In [45]:
flights_short_medium_a.query("cancellation_reason == 'Aircraft unavailable'")["line_number"].value_counts()

line_number
209    1
216    1
222    1
210    1
Name: count, dtype: int64

In [46]:
flights_short_medium_b.query("cancellation_reason == 'Aircraft unavailable'")

,flight_date,line_number,aircraft_id,bookings_total,scheduled_departure,scheduled_arrival,actual_departure,actual_arrival,cancelled,cancellation_reason,delay_reason_dep,delay_reason_arr,seat_capacity,distance_km


In [47]:
flights_short_medium_b.query("cancellation_reason == 'Aircraft unavailable'")["line_number"].value_counts()

Series([], Name: count, dtype: int64)

In [48]:
flights_long.query("cancellation_reason == 'Aircraft unavailable'")

,flight_date,line_number,aircraft_id,bookings_total,scheduled_departure,scheduled_arrival,actual_departure,actual_arrival,cancelled,cancellation_reason,delay_reason_dep,delay_reason_arr,seat_capacity,distance_km
259,2022-01-13,239,AC3000,214,2022-01-13 11:20:37,2022-01-13 22:29:38.326841,NaT,NaT,True,Aircraft unavailable,None,None,270,9798
1139,2022-02-26,240,AC3003,184,2022-02-26 17:20:19,2022-02-27 03:51:26.150565,NaT,NaT,True,Aircraft unavailable,None,None,270,9366
1159,2022-02-27,241,AC3011,252,2022-02-27 16:58:35,2022-02-28 02:53:21.107133,NaT,NaT,True,Aircraft unavailable,None,None,317,8690
13019,2023-10-13,236,AC3005,294,2023-10-13 02:46:14,2023-10-13 13:09:16.566395,NaT,NaT,True,Aircraft unavailable,None,None,317,9154


In [49]:
flights_long.query("cancellation_reason == 'Aircraft unavailable'")["line_number"].value_counts()

line_number
239    1
240    1
241    1
236    1
Name: count, dtype: int64

In [50]:
df_flights = pd.concat([flights_short_medium_a.copy(), flights_short_medium_b.copy(), flights_long.copy()]).reset_index(drop=True)

padding_width = len(str(len(df_flights)))
df_flights.insert(0, "flight_number", "FL1" + df_flights.index.astype(str).str.zfill(padding_width))

df_flights

,flight_number,flight_date,line_number,aircraft_id,bookings_total,scheduled_departure,scheduled_arrival,actual_departure,actual_arrival,cancelled,cancellation_reason,delay_reason_dep,delay_reason_arr,seat_capacity,distance_km
0,FL1000000,2022-01-01,154,AC1032,90,2022-01-01 01:00:00.000000,2022-01-01 01:31:58.226330,2022-01-01 01:13:00.000000,2022-01-01 01:46:58.226330,False,None,None,None,145,300
1,FL1000001,2022-01-01,186,AC1033,94,2022-01-01 01:00:00.000000,2022-01-01 01:35:28.169815,2022-01-01 01:04:00.000000,2022-01-01 01:44:28.169815,False,None,None,None,145,304
2,FL1000002,2022-01-01,164,AC1034,80,2022-01-01 01:00:00.000000,2022-01-01 03:16:39.039647,2022-01-01 01:15:00.000000,2022-01-01 03:30:39.039647,False,None,None,None,145,1838
3,FL1000003,2022-01-01,101,AC1030,88,2022-01-01 01:00:00.000000,2022-01-01 01:59:30.197172,2022-01-01 01:07:00.000000,2022-01-01 02:11:30.197172,False,None,None,None,145,654
4,FL1000004,2022-01-01,102,AC1016,125,2022-01-01 01:00:00.000000,2022-01-01 01:37:00.057670,NaT,NaT,True,Technical failure,None,None,180,431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289339,FL1289339,2024-12-31,229,AC3003,240,2024-12-31 23:52:44.012716,2025-01-01 11:00:58.073750,2024-12-31 23:58:44.012716,2025-01-01 11:05:58.073750,False,None,None,None,270,9798
289340,FL1289340,2024-12-31,230,AC3004,230,2024-12-31 13:04:49.287791,2024-12-31 23:41:50.143004,2024-12-31 13:15:49.287791,2024-12-31 23:54:50.143004,False,None,None,None,270,9366
289341,FL1289341,2024-12-31,231,AC3005,256,2024-12-31 11:13:02.154258,2024-12-31 21:03:49.757605,2024-12-31 11:39:02.154258,2024-12-31 21:30:49.757605,False,None,Technical issue,Late departure,317,8690
289342,FL1289342,2024-12-31,232,AC3009,294,2024-12-31 11:47:44.764324,2025-01-01 06:24:49.942215,2024-12-31 11:48:44.764324,2025-01-01 06:26:49.942215,False,None,None,None,317,16496


In [51]:
df_flights["aircraft_id"].value_counts() / sim_days

aircraft_id
AC1032    6.995438
AC1034    6.995438
AC1017    6.329380
AC1030    6.328467
AC1033    5.889599
            ...   
AC3011    1.451642
AC3006    1.381387
AC3007    1.380474
AC3008    1.320255
AC3010    1.082117
Name: count, Length: 68, dtype: float64

In [52]:
round(df_flights[df_flights["cancellation_reason"] == "Weather"].shape[0] / df_flights.shape[0] * 100, 1)

2.6

In [53]:
round(df_flights[df_flights["cancelled"] == True].shape[0] / df_flights.shape[0] * 100, 1)

4.2

In [54]:
df_flights["cancellation_reason"].value_counts()

cancellation_reason
Weather                 7502
Technical failure       2731
Crew unavailable        1458
Airport closure          475
Aircraft unavailable       8
Name: count, dtype: int64

In [55]:
df_flights["delay_reason_dep"].notnull().sum()

64018

In [56]:
df_flights["delay_reason_arr"].notnull().sum()

96953

In [57]:
df_flights["delay_reason_dep"].value_counts()

delay_reason_dep
Technical issue        21882
Weather                20592
Crew delay             12907
Air traffic control     8637
Name: count, dtype: int64

In [58]:
df_flights["delay_reason_arr"].value_counts()

delay_reason_arr
Late departure         56522
Weather                16160
Airspace congestion    14601
Ground handling         9670
Name: count, dtype: int64

In [59]:
occupancy_rates = df_flights[["flight_number", "line_number", "seat_capacity", "bookings_total"]].copy()
occupancy_rates["occupancy_rate"] = (occupancy_rates["bookings_total"] * 100 / occupancy_rates["seat_capacity"]).round(2)

bins = [0, 60, 70, 75, 85, 90, 100, 200]
labels = ["<60%", "60–70%", "70–75%", "75–85%", "85–90%", "90–100%", ">100%"]
occupancy_rates["occupancy_group"] = pd.cut(occupancy_rates["occupancy_rate"], bins=bins, labels=labels)

occupancy_share_group = (occupancy_rates["occupancy_group"].value_counts(sort=False) * 100 /
                         occupancy_rates["occupancy_group"].value_counts().sum()).astype(int).rename("flights_share")
print(occupancy_rates["occupancy_group"].value_counts(sort=False).rename("flight_count").to_frame().join(occupancy_share_group))

                 flight_count  flights_share
occupancy_group                             
<60%                    10326              3
60–70%                  67329             23
70–75%                  54495             18
75–85%                  94404             32
85–90%                  30227             10
90–100%                 32563             11
>100%                       0              0


In [60]:
occupancy_rates_routes = occupancy_rates.groupby("line_number")["occupancy_rate"].mean()
occupancy_rates_routes.sort_values()

line_number
189    67.955579
132    67.958672
135    68.016104
147    68.031296
199    68.074516
         ...    
239    90.082728
227    92.551578
232    92.598221
214    92.612901
229    92.675018
Name: occupancy_rate, Length: 142, dtype: float64

#### Post-Processing Occupancy (user preference dependent)

we prefer the 70–75% occupancy bucket to be slightly larger than the 60-70% bucket, requiring a small adjustment

In [61]:
np.random.seed(50)
# Step 1: Identify the 60–70% tier
mask_60_70 = occupancy_rates["occupancy_rate"].between(60, 70)

# Step 2: Sample a fraction of those flights — adjust `frac` as needed (e.g., 0.4 = 40%)
indices_60_70 = occupancy_rates[mask_60_70].sample(frac=0.4, random_state=50).index

# Step 3: Apply a modest upward multiplier (e.g., 1.05 to 1.12), capped at seat capacity
multipliers = np.random.uniform(1.05, 1.12, size=len(indices_60_70))

# Get the affected rows from df_flights
affected = df_flights.loc[indices_60_70]

# Step 4: Apply the multiplier
adjusted_bookings = (affected["bookings_total"] * multipliers).round().astype(int)

# Step 5: Ensure we don’t exceed seat capacity
adjusted_bookings = np.minimum(adjusted_bookings, affected["seat_capacity"])

# Step 6: Update the original DataFrame
df_flights.loc[indices_60_70, "bookings_total"] = adjusted_bookings

In [62]:
occupancy_rates = df_flights[["flight_number", "line_number", "seat_capacity", "bookings_total"]].copy()
occupancy_rates["occupancy_rate"] = (occupancy_rates["bookings_total"] * 100 / occupancy_rates["seat_capacity"]).round(2)

bins = [0, 60, 70, 75, 85, 90, 100, 200]
labels = ["<60%", "60–70%", "70–75%", "75–85%", "85–90%", "90–100%", ">100%"]
occupancy_rates["occupancy_group"] = pd.cut(occupancy_rates["occupancy_rate"], bins=bins, labels=labels)

occupancy_share_group = (occupancy_rates["occupancy_group"].value_counts(sort=False) * 100 /
                         occupancy_rates["occupancy_group"].value_counts().sum()).astype(int).rename("flights_share")
print(occupancy_rates["occupancy_group"].value_counts(sort=False).rename("flight_count").to_frame().join(occupancy_share_group))

                 flight_count  flights_share
occupancy_group                             
<60%                     9909              3
60–70%                  49929             17
70–75%                  68218             23
75–85%                  98498             34
85–90%                  30227             10
90–100%                 32563             11
>100%                       0              0


In [63]:
occupancy_rates_routes = occupancy_rates.groupby("line_number")["occupancy_rate"].mean()
occupancy_rates_routes.sort_values()

line_number
135    69.012842
132    69.017582
189    69.070607
199    69.084653
147    69.151811
         ...    
239    90.082728
227    92.551578
232    92.598221
214    92.612901
229    92.675018
Name: occupancy_rate, Length: 142, dtype: float64

In [64]:
df_flights.query("bookings_total > seat_capacity")

,flight_number,flight_date,line_number,aircraft_id,bookings_total,scheduled_departure,scheduled_arrival,actual_departure,actual_arrival,cancelled,cancellation_reason,delay_reason_dep,delay_reason_arr,seat_capacity,distance_km


## Create Flight Capacity by Class and Class Cost Shares Tables


- the flight_capacity_by_class table will contain passenger class capacities for each flight,
  depending on aircraft size and route distance (function: allocate_class_capacities) and
  the number of bookings per class, which are calculated by distributing the (booked) passengers,
  that were assigned by create_flights (function: allocate_passengers_by_class)
- the flight_class_cost_shares table will contain an estimation of how much of each flight's total cost
  can be attributed to each passenger class available for that flight (function: compute_cost_shares)

In [65]:
def allocate_class_capacities(total_capacity_, route_distance_km):
    if total_capacity_ < 120:
        return {"Economy": total_capacity, "Business": 0, "First": 0}
    elif route_distance_km < 2000:
        econ = int(total_capacity_ * 0.9)
        bus = total_capacity_ - econ
        return {"Economy": econ, "Business": bus, "First": 0}
    elif total_capacity_ >= 250:
        econ = int(total_capacity_ * 0.75)
        bus = int(total_capacity_ * 0.2)
        first_ = total_capacity_ - econ - bus
        return {"Economy": econ, "Business": bus, "First": first_}
    else:
        econ = int(total_capacity_ * 0.85)
        bus = total_capacity_ - econ
        return {"Economy": econ, "Business": bus, "First": 0}

In [66]:
def allocate_passengers_by_class(class_capacities, total_passengers_):
    caps = class_capacities.copy()

    # Random draws with binomial suggestions
    p_first = min(np.random.binomial(caps.get("First", 0), 0.9), caps.get("First", 0))
    p_business = min(np.random.binomial(caps.get("Business", 0), 0.85), caps.get("Business", 0))

    # Initial allocation
    allocations = {
        "First": p_first,
        "Business": p_business,
        "Economy": 0
    }

    allocated = p_first + p_business
    remaining = total_passengers_ - allocated

    # Step 1: Try filling Economy with remaining passengers
    eco_cap = caps.get("Economy", 0)
    p_economy = min(remaining, eco_cap)
    allocations["Economy"] = p_economy
    remaining -= p_economy

    # Step 2: If still underfilled, redistribute proportionally to all classes with space
    if remaining > 0:
        for cls_ in ["Economy", "Business", "First"]:  # Economy-first fill
            space = caps[cls_] - allocations[cls_]
            add = min(space, remaining)
            allocations[cls_] += add
            remaining -= add
            if remaining == 0:
                break

    # Step 3: If overfilled, reduce proportionally
    total_assigned = sum(allocations.values())
    overflow = total_assigned - total_passengers_
    if overflow > 0:
        # Reduce proportionally from classes that were over-allocated
        for cls_ in ["First", "Business", "Economy"]:
            if allocations[cls_] > 0:
                reduction = int(round((allocations[cls_] / total_assigned) * overflow))
                allocations[cls_] = max(0, allocations[cls_] - reduction)
        # Final correction to match total exactly
        diff = sum(allocations.values()) - total_passengers_
        if diff > 0:
            # Remove 1 passenger at a time from the largest class until it matches
            for cls_ in sorted(allocations, key=lambda x: -allocations[x]):
                while allocations[cls_] > 0 and diff > 0:
                    allocations[cls_] -= 1
                    diff -= 1
                    if diff == 0:
                        break

    return allocations

In [67]:
def compute_cost_shares(class_caps_, cost_weights_):
    weighted = {
        cls_: seats * cost_weights_[cls_]
        for cls_, seats in class_caps_.items() if seats > 0
    }
    total_weight = sum(weighted.values())
    return {
        cls_: round(weight / total_weight, 2)
        for cls_, weight in weighted.items()
    }

In [68]:
flight_class_records = []

cost_weights = {"Economy": 1.0, "Business": 2.25, "First": 3.5}
cost_share_records = []

np.random.seed(50)

for _, row in df_flights.iterrows():
    flight_number = row["flight_number"]
    flight_date = row["flight_date"]
    aircraft_id = row["aircraft_id"]
    total_bookings = row["bookings_total"]
    route_distance = row["distance_km"]
    total_capacity = row["seat_capacity"]

    class_caps = allocate_class_capacities(total_capacity, route_distance)
    class_passengers = allocate_passengers_by_class(class_caps, total_bookings)
    cost_shares = compute_cost_shares(class_caps, cost_weights)

    for cls in class_caps:
        flight_class_records.append({
            "flight_number": flight_number,
            "flight_date": flight_date,
            "class_name": cls,
            "capacity": class_caps[cls],
            "class_bookings": class_passengers[cls],
        })

        # Only include cost share if capacity > 0
        if class_caps[cls] > 0:
            cost_share_records.append({
                "flight_number": flight_number,
                "flight_date": flight_date,
                "class_name": cls,
                "cost_share": cost_shares[cls],
            })

In [69]:
df_flight_capacity_by_class = pd.DataFrame(flight_class_records)
df_flight_capacity_by_class

,flight_number,flight_date,class_name,capacity,class_bookings
0,FL1000000,2022-01-01,Economy,130,76
1,FL1000000,2022-01-01,Business,15,14
2,FL1000000,2022-01-01,First,0,0
3,FL1000001,2022-01-01,Economy,130,81
4,FL1000001,2022-01-01,Business,15,13
...,...,...,...,...,...
868027,FL1289342,2024-12-31,Business,63,54
868028,FL1289342,2024-12-31,First,17,15
868029,FL1289343,2024-12-31,Economy,237,180
868030,FL1289343,2024-12-31,Business,63,54


In [70]:
# verify that the sum of class bookings per flight matches the (booked) passenger totals in df_flights
capacity_grp = df_flight_capacity_by_class.groupby(["flight_number", "flight_date"])["class_bookings"].sum().reset_index()
comparison = df_flights[["flight_number", "flight_date", "bookings_total"]].merge(capacity_grp, on=["flight_number", "flight_date"])

comparison.shape[0], comparison.query("bookings_total != class_bookings").shape[0]

(289344, 0)

In [71]:
# verify that bookings don't exceed capacity
df_flight_capacity_by_class.query("capacity < class_bookings")

,flight_number,flight_date,class_name,capacity,class_bookings


In [72]:
df_flight_class_cost_shares = pd.DataFrame(cost_share_records)

df_flight_class_cost_shares.insert(
    0, "cost_share_id",
    "CS_" + df_flight_class_cost_shares["flight_number"].str[2:] +
    "_" + df_flight_class_cost_shares["class_name"].str[0]
)

df_flight_class_cost_shares

,cost_share_id,flight_number,flight_date,class_name,cost_share
0,CS_1000000_E,FL1000000,2022-01-01,Economy,0.79
1,CS_1000000_B,FL1000000,2022-01-01,Business,0.21
2,CS_1000001_E,FL1000001,2022-01-01,Economy,0.79
3,CS_1000001_B,FL1000001,2022-01-01,Business,0.21
4,CS_1000002_E,FL1000002,2022-01-01,Economy,0.79
...,...,...,...,...,...
579769,CS_1289342_B,FL1289342,2024-12-31,Business,0.32
579770,CS_1289342_F,FL1289342,2024-12-31,First,0.14
579771,CS_1289343_E,FL1289343,2024-12-31,Economy,0.54
579772,CS_1289343_B,FL1289343,2024-12-31,Business,0.32


In [14]:
if csv_settings == "imm":
    df_flight_capacity_by_class.to_csv(csv_folder_path + "flight_capacity_by_class.csv", index=False, sep=separator, decimal=dec)
    df_flight_class_cost_shares.to_csv(csv_folder_path + "flight_class_cost_shares", index=False, sep=separator, decimal=dec)

## Prepare Bookings Table

- the bookings table will be generated by expanding flight_capacity_by_class based on the
  total number of class bookings, resulting in a dataframe with one row per booking
- other columns will be added in multiple stages, since some of them require the customers
  table to be created first, in order to allow for noteworthy correlations between customer
  attributes and booking behavior

In [73]:
flight_caps_merged = df_flight_capacity_by_class.merge(df_flights[["flight_number", "line_number", "cancellation_reason"]], on=["flight_number"], how="left")
flight_caps_merged = flight_caps_merged.merge(df_routes[["line_number", "price_economy", "price_business", "price_first"]], on=["line_number"], how="left")

In [74]:
flight_rows_expanded = flight_caps_merged.loc[
    flight_caps_merged.index.repeat(flight_caps_merged["class_bookings"])
].reset_index(drop=True)

bookings = flight_rows_expanded.copy()
#bookings

- the following function generates customer ids and assigns them to bookings, while ensuring
  that the same customer doesn't end up booking multiple flights that occur at the same time
- the customer ids will be subsequently used to generate the customers table
- the multiplier in the n_required_customers variable below determines the number of unique customers
  relative to the number of bookings; when optimizing for realism, the multiplier should be set to
  a value close to 0.7; for performance reasons a lower value than that is recommended; the cost for
  the (significant) performance boost is a somewhat inflated number of frequent flyers

In [75]:
def assign_customers(bookings_: pd.DataFrame, n_customers_: int) -> pd.DataFrame:
    bookings_ = bookings_.copy()

    # Step 1: Ensure sorted to keep reproducibility and group consistency
    bookings_.sort_values("flight_date", inplace=True)
    bookings_.reset_index(drop=True, inplace=True)

    # Step 2: Get counts per day
    date_counts = bookings_["flight_date"].value_counts().to_dict()

    # Step 3: Track which customer IDs are available per day
    customer_ids_pool = np.arange(1, n_customers_ + 1)
    assigned_ids = []

    for date in bookings_["flight_date"].unique():
        n_needed = date_counts[date]

        # Error out if impossible to assign unique customers
        if n_needed > n_customers_:
            raise ValueError(f"Cannot assign {n_needed} unique customers on {date} with only {n_customers_} total.")

        # Randomly sample unique customer IDs for this date
        sampled_ids = np.random.choice(customer_ids_pool, size=n_needed, replace=False)

        # Add to the global list
        assigned_ids.extend(sampled_ids)

    # Step 4: Assign to DataFrame
    bookings_["customer_id"] = assigned_ids

    return bookings_

In [76]:
# multiplier determines the number of customers we'll end up with relative to the number of bookings
n_required_customers = int(len(bookings) * 0.34)

np.random.seed(50)

# Assign customer IDs
bookings = assign_customers(bookings, n_required_customers)
#bookings

In [77]:
duplicates = bookings.duplicated(subset=["customer_id", "flight_date"], keep=False)
# should return 0 rows
bookings[duplicates]

,flight_number,flight_date,class_name,capacity,class_bookings,line_number,cancellation_reason,price_economy,price_business,price_first,customer_id


- a realistic assignment of frequent flyer statuses would require something like a point score system
  that updates statuses after every flight, significantly increasing calculation time; some customers
  would also start off with randomly determined point scores reflecting flights that occurred prior to
  the simulation period
- the drastically simplified approach chosen here determines frequent flyer status based on a customer's
  total booked flights in the earliest simulated year and applies it across the entire simulation period

In [78]:
status_map = bookings[["flight_date", "customer_id"]]
customer_earliest_year = status_map.groupby("customer_id")["flight_date"].min().dt.year.rename("customer_earliest_year")
status_map = status_map.merge(customer_earliest_year, on="customer_id")
status_map = status_map.groupby(["customer_id", "customer_earliest_year"]).count().rename(columns={"flight_date": "n_bookings"}).reset_index()

thresholds = [3, 5, 7, 10]
yearly_flights_booked = [status_map["n_bookings"] <= thresholds[0], status_map["n_bookings"] <= thresholds[1],
                         status_map["n_bookings"] <= thresholds[2], status_map["n_bookings"] <= thresholds[3]]
frequent_flyer_statuses = ["0-N", "1-B", "2-S", "3-G"]
status_map["frequent_flyer_status_code"] = np.select(yearly_flights_booked, frequent_flyer_statuses, default="4-P")

status_map

customer_id
13174237    4-P
12958175    4-P
7483590     4-P
649339      4-P
112922      4-P
           ... 
8625212     0-N
7592733     0-N
4578541     0-N
6997473     0-N
12721160    0-N
Name: frequent_flyer_status_code, Length: 13087529, dtype: object

In [79]:
status_map.value_counts()

frequent_flyer_status_code
0-N    8395201
1-B    3617645
2-S     928728
3-G     142732
4-P       3223
Name: count, dtype: int64

In [15]:
if csv_settings == "imm":
    status_map.reset_index().to_csv(csv_folder_path + "flyer_status_map.csv", index=False)

In [78]:
# only needed if the next code cell is executed multiple times without resetting bookings df first
#bookings.drop("frequent_flyer_status_code", axis=1, inplace=True)

In [80]:
bookings = bookings.merge(status_map, on="customer_id", how="left")

- simplified frequent flyer benefit system: customers receive a discount on flight prices corresponding to their frequent flyer tier

In [81]:
# key names other than None can be changed, values other than 0 can be changed if kept in ascending order;
# if changes to the order, length or the first key-value-pair are made, corresponding changes need to be
# made in the map_to_status function and in the creation of the frequent flyer discounts table
discounts = {
    "0-N": 0,
    "1-B": 1.5,
    "2-S": 3.0,
    "3-G": 8.0,
    "4-P": 13.0,
}

bookings["frequent_flyer_discount_pct"] = bookings["frequent_flyer_status_code"].map(discounts)
#bookings

In [82]:
bookings["price_paid"] = np.select(
    [
        bookings["class_name"] == "Economy",
        bookings["class_name"] == "Business",
        bookings["class_name"] == "First"
    ],
    [
        round(bookings["price_economy"] * (1 - bookings["frequent_flyer_discount_pct"] / 100), 2),
        round(bookings["price_business"] * (1 - bookings["frequent_flyer_discount_pct"] / 100), 2),
        round(bookings["price_first"] * (1 - bookings["frequent_flyer_discount_pct"] / 100), 2),
    ],
    default=None
)

bookings

,flight_number,flight_date,class_name,capacity,class_bookings,line_number,cancellation_reason,price_economy,price_business,price_first,customer_id,frequent_flyer_status_code,frequent_flyer_discount_pct,price_paid
0,FL1000000,2022-01-01,Economy,130,76,154,None,34.59,87.92,174.46,10169503,1-B,1.5,34.07
1,FL1133817,2022-01-01,Economy,130,76,171,None,120.81,318.03,607.60,1407797,1-B,1.5,119.0
2,FL1133817,2022-01-01,Economy,130,76,171,None,120.81,318.03,607.60,12662126,0-N,0.0,120.81
3,FL1133817,2022-01-01,Economy,130,76,171,None,120.81,318.03,607.60,6545603,0-N,0.0,120.81
4,FL1133817,2022-01-01,Economy,130,76,171,None,120.81,318.03,607.60,924261,0-N,0.0,120.81
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40633004,FL1133669,2024-12-31,Business,15,12,150,None,39.88,104.21,196.01,9998191,1-B,1.5,102.65
40633005,FL1133669,2024-12-31,Business,15,12,150,None,39.88,104.21,196.01,4962859,1-B,1.5,102.65
40633006,FL1133669,2024-12-31,Business,15,12,150,None,39.88,104.21,196.01,7525679,0-N,0.0,104.21
40633007,FL1133669,2024-12-31,Business,15,12,150,None,39.88,104.21,196.01,6324710,2-S,3.0,101.08


In [83]:
# placeholder: updated after df_customers is created

bookings["booking_time"] = pd.Timestamp(year=2021, month=1, day=1, hour=1, minute=1, second=1)
#bookings

In [84]:
# placeholder: updated after df_customers is created
bookings["checked_in"] = True

In [85]:
# placeholder: updated after df_customers is created
bookings["flight_cxl_refund"] = False

In [86]:
df_bookings = bookings[[
    "customer_id",
    "flight_number",
    "flight_date",
    "booking_time",
    "class_name",
    "frequent_flyer_status_code",
    "price_paid",
    "checked_in",
    "flight_cxl_refund",
    "cancellation_reason",
    "class_bookings",
    "capacity"
]].copy()

df_bookings.insert(0, "booking_id", df_bookings.index + 1)
df_bookings

,booking_id,customer_id,flight_number,flight_date,booking_time,class_name,frequent_flyer_status_code,price_paid,checked_in,flight_cxl_refund,cancellation_reason,class_bookings,capacity
0,1,10169503,FL1000000,2022-01-01,2021-01-01 01:01:01,Economy,1-B,34.07,True,False,None,76,130
1,2,1407797,FL1133817,2022-01-01,2021-01-01 01:01:01,Economy,1-B,119.0,True,False,None,76,130
2,3,12662126,FL1133817,2022-01-01,2021-01-01 01:01:01,Economy,0-N,120.81,True,False,None,76,130
3,4,6545603,FL1133817,2022-01-01,2021-01-01 01:01:01,Economy,0-N,120.81,True,False,None,76,130
4,5,924261,FL1133817,2022-01-01,2021-01-01 01:01:01,Economy,0-N,120.81,True,False,None,76,130
...,...,...,...,...,...,...,...,...,...,...,...,...,...
40633004,40633005,9998191,FL1133669,2024-12-31,2021-01-01 01:01:01,Business,1-B,102.65,True,False,None,12,15
40633005,40633006,4962859,FL1133669,2024-12-31,2021-01-01 01:01:01,Business,1-B,102.65,True,False,None,12,15
40633006,40633007,7525679,FL1133669,2024-12-31,2021-01-01 01:01:01,Business,0-N,104.21,True,False,None,12,15
40633007,40633008,6324710,FL1133669,2024-12-31,2021-01-01 01:01:01,Business,2-S,101.08,True,False,None,12,15


In [87]:
len(df_bookings["customer_id"].unique().tolist())

13087529

In [16]:
if csv_settings == "imm":
    df_bookings.to_csv(csv_folder_path + "temp_bookings.csv", index=False, sep=separator, decimal=dec)

## Create Customers Table

- gender and date of birth are assigned in a way that makes these customer attributes to some degree
  predictive of passenger class preference and frequent flyer tier representation

In [88]:
# Seed for reproducibility
Faker.seed(50)
random.seed(50)
np.random.seed(50)

# Country and localization setup
country_weights_dict = {
    "Germany": 25, "United Kingdom": 12, "France": 12, "Austria": 6, "Spain": 4, "Italy": 2.5,
    "Sweden": 2.5, "Portugal": 2, "Netherlands": 2, "Switzerland": 2, "Denmark": 2, "Finland": 2,
    "Norway": 2, "Poland": 2, "United States": 4, "Canada": 1, "Brazil": 1,
    "Australia": 0.5
}

countries = list(country_weights_dict.keys())
country_weights = list(country_weights_dict.values())

locale_map = {
    "Germany": "de_DE", "United Kingdom": "en_GB", "France": "fr_FR", "Austria": "de_AT",
    "Spain": "es_ES", "Italy": "it_IT", "Sweden": "sv_SE", "Portugal": "pt_PT",
    "Netherlands": "nl_NL", "Switzerland": "de_CH", "Denmark": "da_DK", "Finland": "fi_FI",
    "Norway": "no_NO", "Poland": "pl_PL", "United States": "en_US", "Canada": "en_CA",
    "Brazil": "pt_BR", "Australia": "en_AU"
}

locale_fakers = {loc: Faker(loc) for loc in set(locale_map.values())}
default_fake = Faker("en_US")

def sanitize(s):
    return re.sub(r"[^a-zA-Z0-9]", "", unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()).lower()

# Get the most frequent class/status per customer (fast + simple)
mode_class = df_bookings.groupby("customer_id")["class_name"].agg(lambda x: x.value_counts().idxmax())
mode_status = df_bookings.groupby("customer_id")["frequent_flyer_status_code"].agg(lambda x: x.value_counts().idxmax())


customer_profiles = pd.DataFrame({
    "class_name": mode_class,
    "frequent_flyer_status_code": mode_status
}).reset_index()

# Gender assignment
def assign_gender(status_code, class_name):
    key = (status_code, class_name)
    base_gender_probs = {
        ("4-P", "First"): [0.75, 0.25],
        ("3-G", "First"): [0.7, 0.3],
        ("3-G", "Business"): [0.65, 0.35],
        ("2-S", "Business"): [0.6, 0.4],
        ("0-N", "Economy"): [0.49, 0.51]
    }

    male_p, female_p = base_gender_probs.get(key, [0.55, 0.45])
    return np.random.choice(["male", "female"], p=[male_p, female_p])

# Date of birth assignment
def assign_dob(status_code, class_name):
    key = (status_code, class_name)
    base_age_ranges = {
        ("4-P", "First"): (45, 75),
        ("3-G", "Business"): (40, 70),
        ("2-S", "Business"): (35, 65),
        ("1-B", "Economy"): (25, 50),
        ("0-N", "Economy"): (18, 40)
    }

    min_age, max_age = base_age_ranges.get(key, (20, 80))
    base_age = np.random.uniform(min_age, max_age)

    # Convert age in years to days
    age_in_days = int(base_age * 365.25)
    today_ = pd.Timestamp.now()
    random_day_offset = random.randint(-182, 182)
    dob_ = today_ - pd.Timedelta(days=age_in_days + random_day_offset)

    return dob_

new_customers = []

for row in customer_profiles.itertuples():
    cid = row.customer_id
    cls = row.class_name
    status_ = row.frequent_flyer_status_code

    gender = assign_gender(status_, cls)

    nationality = random.choices(countries, weights=country_weights, k=1)[0]
    dob = assign_dob(status_, cls)

    locale = locale_map.get(nationality, "en_US")
    faker = locale_fakers.get(locale, default_fake)

    first = faker.first_name_male() if gender == "male" else faker.first_name_female()
    last = faker.last_name()
    email = f"{sanitize(first)}.{sanitize(last)}@{default_fake.free_email_domain()}"
    phone = re.sub(r"\D", "", default_fake.phone_number())[:10]

    new_customers.append({
        "customer_id": cid,
        "full_name": f"{first} {last}",
        "email": email,
        "phone": phone,
        "date_of_birth": dob,
        "nationality": nationality,
        "gender": gender,
        "frequent_flyer_status_code": status_
    })

df_customers = pd.DataFrame(new_customers)

df_customers["date_of_birth"] = pd.to_datetime(df_customers["date_of_birth"])

df_customers

,customer_id,full_name,email,phone,date_of_birth,nationality,gender,frequent_flyer_status_code
0,1,Zoé Da Costa,zoe.dacosta@yahoo.com,1737751853,2002-10-03 20:59:15.975115,France,female,0-N
1,2,Jamie Barrett,jamie.barrett@yahoo.com,1896311956,1999-02-01 20:59:15.982120,United Kingdom,male,0-N
2,3,Cornelio Salas,cornelio.salas@yahoo.com,0013979249,1985-10-05 20:59:15.982120,Spain,male,0-N
3,4,Heinrich Plath,heinrich.plath@hotmail.com,1827998415,1990-09-20 20:59:15.983624,Germany,male,0-N
4,5,Zita Zobel,zita.zobel@hotmail.com,1732291246,2000-07-18 20:59:15.983624,Germany,female,0-N
...,...,...,...,...,...,...,...,...
13087524,13815218,Andrew Metcalfe,andrew.metcalfe@hotmail.com,6536033514,1963-05-29 21:27:25.758252,United Kingdom,male,1-B
13087525,13815219,Araceli Amor,araceli.amor@yahoo.com,1560689131,1992-01-29 21:27:25.758252,Spain,female,1-B
13087526,13815220,Damien Jones,damien.jones@gmail.com,2068735804,1995-11-21 21:27:25.758252,United Kingdom,male,0-N
13087527,13815221,Gerold Lange,gerold.lange@yahoo.com,1296953768,1991-05-14 21:27:25.758252,Germany,male,0-N


#### Investigating Results

In [89]:
df_customers["nationality"].value_counts()

nationality
Germany           3871589
France            1858712
United Kingdom    1857669
Austria            929066
Spain              620165
United States      619109
Sweden             388388
Italy              387058
Denmark            310452
Portugal           310386
Netherlands        310214
Norway             310046
Poland             309511
Switzerland        309463
Finland            309432
Canada             154871
Brazil             154307
Australia           77091
Name: count, dtype: int64

In [90]:
df_customers["gender"].value_counts() * 100 / len(df_customers)

gender
male      51.569452
female    48.430548
Name: count, dtype: float64

In [91]:
df_customers.groupby("gender")["frequent_flyer_status_code"].value_counts(normalize=True).sort_index() * 100

gender  frequent_flyer_status_code
female  0-N                           66.687229
        1-B                           25.708645
        2-S                            6.569931
        3-G                            1.011776
        4-P                            0.022419
male    0-N                           61.760555
        1-B                           29.457532
        2-S                            7.590596
        3-G                            1.164618
        4-P                            0.026700
Name: proportion, dtype: float64

In [92]:
df_customers.merge(df_bookings[["customer_id", "class_name"]], on="customer_id").groupby("gender")["class_name"].value_counts(normalize=True).sort_index() * 100

gender  class_name
female  Business      13.209257
        Economy       86.052554
        First          0.738188
male    Business      13.732862
        Economy       85.499301
        First          0.767837
Name: proportion, dtype: float64

In [93]:
print(df_customers["date_of_birth"].dt.year.min(), df_customers["date_of_birth"].dt.year.max(), df_customers["date_of_birth"].dt.year.mean().astype(int), df_customers["date_of_birth"].dt.year.median())

1945 2008 1990 1992.0


In [94]:
test_customers = df_customers.copy()

# Calculate age
today = pd.to_datetime('today')
test_customers['age'] = (today - test_customers['date_of_birth']).dt.days // 365

bins = [0, 18, 25, 35, 45, 60, 75, 100]
labels = ['<18', '18–24', '25–34', '35–44', '45–59', '60–74', '75+']
test_customers['age_group'] = pd.cut(test_customers['age'], bins=bins, labels=labels, right=False)

test_customers['age_group'].value_counts().sort_index()

age_group
<18        40337
18–24    2524731
25–34    5118079
35–44    3453273
45–59    1238010
60–74     533680
75+       179419
Name: count, dtype: int64

In [95]:
test_customers['age_group'].value_counts(normalize=True).sort_index() * 100

age_group
<18       0.308209
18–24    19.291121
25–34    39.106534
35–44    26.385982
45–59     9.459463
60–74     4.077775
75+       1.370916
Name: proportion, dtype: float64

In [96]:
test_customers.groupby("age_group", observed=True)["frequent_flyer_status_code"].value_counts(normalize=True).sort_index() * 100

age_group  frequent_flyer_status_code
<18        0-N                           100.000000
18–24      0-N                            95.399233
           1-B                             1.105781
           2-S                             3.012162
           3-G                             0.471377
           4-P                             0.011447
25–34      0-N                            69.338965
           1-B                            27.218611
           2-S                             2.975550
           3-G                             0.456714
           4-P                             0.010160
35–44      0-N                            53.835883
           1-B                            40.923147
           2-S                             4.539375
           3-G                             0.685002
           4-P                             0.016593
45–59      0-N                            18.562936
           1-B                            59.388131
           2-S            

In [97]:
test_customers.merge(df_bookings[["customer_id", "class_name"]], on="customer_id").groupby("age_group", observed=True)["class_name"].value_counts(normalize=True).sort_index() * 100

age_group  class_name
<18        Business       8.077284
           Economy       91.475330
           First          0.447386
18–24      Business      10.331211
           Economy       89.082016
           First          0.586773
25–34      Business      11.354701
           Economy       87.980243
           First          0.665056
35–44      Business      12.540530
           Economy       86.738372
           First          0.721098
45–59      Business      17.917984
           Economy       81.118300
           First          0.963716
60–74      Business      25.441028
           Economy       73.303599
           First          1.255373
75+        Business      25.170354
           Economy       73.595444
           First          1.234202
Name: proportion, dtype: float64

In [17]:
if csv_settings == "imm":
    df_customers.to_csv(csv_folder_path + "customers_before_noise.csv", index=False, sep=separator, decimal=dec)

## Update Bookings Table

In [98]:
df_bookings_merged = df_bookings.copy().merge(
    df_customers[["customer_id", "date_of_birth", "nationality", "gender"]],
    on="customer_id"
)

- customer age and gender will influence how likely a customer is to miss their check-in

In [99]:
def apply_check_in_behavior(df_: pd.DataFrame, seed: int = None) -> None:
    """
    Adds check-in rate and outcome to the given bookings DataFrame in-place.

    Args:
        df_: A DataFrame containing 'flight_date', 'booked_rate', and 'customer_id'.
        seed: Optional random seed for reproducibility.
    """
    if seed is not None:
        np.random.seed(seed)

    n_ = len(df_)
    booked_rate = df_["booked_rate"].values

    # 1. Base rate from booked rate tier
    base_rate = np.select(
        [
            booked_rate >= 90,
            booked_rate >= 75,
            booked_rate >= 70,
            booked_rate >= 60
        ],
        [
            np.random.uniform(0.98, 0.99, size=n_),
            np.random.uniform(0.96, 0.98, size=n_),
            np.random.uniform(0.94, 0.96, size=n_),
            np.random.uniform(0.92, 0.94, size=n_)
        ],
        default=np.random.uniform(0.90, 0.93, size=n_)
    )

    # 2. Weekday modifier
    weekday = df_["flight_date"].dt.weekday.values
    weekday_mod = np.select(
        [
            weekday == 0,  # Monday
            weekday == 4,  # Friday
            weekday == 6   # Sunday
        ],
        [
            -0.005,
             0.01,
            -0.01
        ],
        default=0.0
    )

    # 3. Demographic modifiers

    # -- Gender effect
    gender_mod = df_["gender"].map({"female": 0.005, "male": -0.005}).fillna(0.0).values

    # -- Age effect
    today_ = pd.Timestamp.now()
    age = ((today_ - pd.to_datetime(df_["date_of_birth"])).dt.days / 365.25).values
    age_mod = np.piecewise(age,
        [
            age < 25,
            (age >= 25) & (age < 30),
            (age >= 30) & (age < 60),
            (age >= 60) & (age <= 70),
            age > 70
        ],
        [
            -0.015,
            -0.005,
             0.010,
             0.005,
            -0.01
        ]
    )

    # 4. Random drop
    random_drop = np.random.beta(2, 20, size=n_) * 0.05

    # 5. Final check-in rate
    check_in_rate = base_rate + weekday_mod + gender_mod + age_mod - random_drop
    check_in_rate = np.clip(check_in_rate, 0.80, 1.0)

    # 6. Assign to original DataFrame
    df_["check_in_rate"] = check_in_rate
    df_["checked_in"] = np.random.rand(n_) < check_in_rate
    df_["missed_check_in_drop"] = ((booked_rate / 100) - check_in_rate).round(4)

In [100]:
df_bookings_merged["booked_rate"] = df_bookings_merged["class_bookings"] * 100 / df_bookings_merged["capacity"]

apply_check_in_behavior(df_bookings_merged, seed=50)

df_bookings_merged["checked_in"].value_counts()

checked_in
True     38933469
False     1699540
Name: count, dtype: int64

- investigate results

In [101]:
df_bookings_merged.groupby("gender")["checked_in"].value_counts(normalize=True).sort_index() * 100

gender  checked_in
female  False          3.693763
        True          96.306237
male    False          4.622637
        True          95.377363
Name: proportion, dtype: float64

In [102]:
test_bookings = df_bookings_merged.copy()

# Calculate age
today = pd.to_datetime('today')
test_bookings['age'] = (today - test_bookings['date_of_birth']).dt.days // 365

bins = [0, 18, 25, 35, 45, 60, 75, 100]
labels = ['<18', '18–24', '25–34', '35–44', '45–59', '60–74', '75+']
test_bookings['age_group'] = pd.cut(test_bookings['age'], bins=bins, labels=labels, right=False)

age_group_counts = test_bookings['age_group'].value_counts().sort_index()
test_bookings.groupby("age_group", observed=True)["checked_in"].value_counts(normalize=True).sort_index() * 100

age_group  checked_in
<18        False          6.015369
           True          93.984631
18–24      False          5.959416
           True          94.040584
25–34      False          4.218083
           True          95.781917
35–44      False          3.479155
           True          96.520845
45–59      False          3.410495
           True          96.589505
60–74      False          4.271606
           True          95.728394
75+        False          5.294871
           True          94.705129
Name: proportion, dtype: float64

- add refunds for cancelled flights

In [103]:
cancelled_mask = df_bookings_merged["cancellation_reason"].notna()
df_bookings_merged.loc[cancelled_mask, "flight_cxl_refund"] = True

df_bookings_merged.loc[cancelled_mask, "checked_in"] = False

df_bookings_merged

,booking_id,customer_id,flight_number,flight_date,booking_time,class_name,frequent_flyer_status_code,price_paid,checked_in,flight_cxl_refund,cancellation_reason,class_bookings,capacity,date_of_birth,nationality,gender,booked_rate,check_in_rate,missed_check_in_drop
0,1,10169503,FL1000000,2022-01-01,2021-01-01 01:01:01,Economy,1-B,34.07,False,False,None,76,130,1994-12-17 21:20:09.821844,Germany,male,58.461538,0.926734,-0.3421
1,2,1407797,FL1133817,2022-01-01,2021-01-01 01:01:01,Economy,1-B,119.0,True,False,None,76,130,1994-11-11 21:02:17.309626,Germany,female,58.461538,0.916740,-0.3321
2,3,12662126,FL1133817,2022-01-01,2021-01-01 01:01:01,Economy,0-N,120.81,True,False,None,76,130,2000-07-19 21:25:02.214333,Germany,male,58.461538,0.890962,-0.3063
3,4,6545603,FL1133817,2022-01-01,2021-01-01 01:01:01,Economy,0-N,120.81,True,False,None,76,130,2004-07-28 21:12:44.491533,Spain,male,58.461538,0.904221,-0.3196
4,5,924261,FL1133817,2022-01-01,2021-01-01 01:01:01,Economy,0-N,120.81,True,False,None,76,130,2004-06-21 21:01:16.661612,United States,female,58.461538,0.894054,-0.3094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40633004,40633005,9998191,FL1133669,2024-12-31,2021-01-01 01:01:01,Business,1-B,102.65,True,False,None,12,15,1977-10-13 21:19:45.491733,Finland,female,80.000000,0.986365,-0.1864
40633005,40633006,4962859,FL1133669,2024-12-31,2021-01-01 01:01:01,Business,1-B,102.65,True,False,None,12,15,1975-10-27 21:09:31.459530,United Kingdom,male,80.000000,0.957173,-0.1572
40633006,40633007,7525679,FL1133669,2024-12-31,2021-01-01 01:01:01,Business,0-N,104.21,True,False,None,12,15,1998-08-02 21:14:42.572742,Finland,female,80.000000,0.972473,-0.1725
40633007,40633008,6324710,FL1133669,2024-12-31,2021-01-01 01:01:01,Business,2-S,101.08,True,False,None,12,15,1993-12-05 21:12:17.952529,Germany,male,80.000000,0.968545,-0.1685


- customer age and gender will influence booking lead times, nationality will influence booking time of day

In [106]:
# Assign booking times

time_profiles = {
    # Central Europe (early–mid-morning preference, some evening activity)
    "Germany":        [0.45, 0.30, 0.20, 0.05],
    "Austria":        [0.40, 0.30, 0.20, 0.10],
    "Switzerland":    [0.35, 0.35, 0.20, 0.10],
    "France":         [0.30, 0.35, 0.25, 0.10],
    "Netherlands":    [0.40, 0.30, 0.20, 0.10],
    "Belgium":        [0.35, 0.35, 0.20, 0.10],  # optional if present
    "Poland":         [0.30, 0.35, 0.25, 0.10],

    # Scandinavia (morning-heavy, fewer evenings)
    "Sweden":         [0.50, 0.30, 0.15, 0.05],
    "Norway":         [0.45, 0.35, 0.15, 0.05],
    "Denmark":        [0.45, 0.35, 0.15, 0.05],
    "Finland":        [0.50, 0.30, 0.15, 0.05],

    # Southern Europe (more evening activity, later mornings)
    "Spain":          [0.15, 0.25, 0.45, 0.15],
    "Italy":          [0.20, 0.30, 0.40, 0.10],
    "Portugal":       [0.20, 0.30, 0.40, 0.10],

    # Anglo countries
    "United Kingdom": [0.35, 0.30, 0.25, 0.10],
    "United States":  [0.25, 0.35, 0.30, 0.10],  # mix of coasts → balanced
    "Canada":         [0.30, 0.35, 0.25, 0.10],
    "Australia":      [0.20, 0.30, 0.35, 0.15],

    # South America
    "Brazil":         [0.15, 0.25, 0.45, 0.15],

    # Fallback
    "Default":        [0.25, 0.25, 0.25, 0.25]
}

time_windows = {
    0: (6, 12),   # Morning
    1: (12, 17),  # Afternoon
    2: (17, 22),  # Evening
    3: (22, 6)    # Night (wrap-around)
}

np.random.seed(50)
today = pd.Timestamp.now()

# --- Step 1: Compute age
df_bookings_merged["age"] = ((today - df_bookings_merged["date_of_birth"]).dt.days // 365).clip(lower=0)

# --- Step 2: Compute mean days before flight
base_mean = 30 + 0.5 * df_bookings_merged["age"]
base_mean = base_mean.clip(lower=10)

# Gender-based shift
gender_shift = np.where(df_bookings_merged["gender"] == "female", 5, 0)
mean_days = base_mean + gender_shift

# --- Dynamic maximum booking days based on age and gender
def compute_max_days(age, gender_):
    # Older people tend to book earlier
    # Females may book slightly earlier on average
    base = 60 + (age * 0.5)  # e.g., 60 + 0.5*age
    if gender_ == "female":
        base += 10  # shift for female
    return min(base, 120)  # impose hard ceiling

df_bookings_merged["max_booking_days"] = df_bookings_merged.apply(lambda row_: compute_max_days(row_["age"], row_["gender"]), axis=1)

# Draw base distribution
days_before = np.random.normal(loc=mean_days, scale=10)

# Cap and clean
days_before = np.minimum(days_before, df_bookings_merged["max_booking_days"])
days_before = np.nan_to_num(days_before, nan=10, posinf=120, neginf=1)
days_before = np.clip(days_before, 0, None).astype(int)  # allow 0 days

# Inject a small chance of same-day booking
same_day_mask = np.random.rand(len(days_before)) < 0.01  # e.g., 1% chance
days_before[same_day_mask] = 0

# --- Step 3: Get the booking date
df_bookings_merged["flight_date"] = pd.to_datetime(df_bookings_merged["flight_date"])
booking_dates = df_bookings_merged["flight_date"] - pd.to_timedelta(days_before, unit="D")

# --- Step 4: Assign hour range per nationality
# If nationality not in dict, fallback to Default
probs = df_bookings_merged["nationality"].map(
    lambda nat: time_profiles.get(nat, time_profiles["Default"])
)

# Stack into (n,4) array
probs_matrix = np.vstack(probs.to_numpy())

# Randomly pick a time window per row
cum_probs = probs_matrix.cumsum(axis=1)
rand = np.random.rand(len(df_bookings_merged), 1)
# noinspection PyUnresolvedReferences
choices = (rand < cum_probs).argmax(axis=1)

# Translate chosen window → hour ranges
starts = np.array([time_windows[c][0] for c in choices])
ends   = np.array([time_windows[c][1] for c in choices])

# Sample booking hours row by row
hours = np.array([
    np.random.randint(start, end) if start < end else np.random.randint(0, 24)
    for start, end in zip(starts, ends)
])

# Add random minutes/seconds
minutes = np.random.randint(0, 60, len(hours))
seconds = np.random.randint(0, 60, len(hours))

# Build booking_time
time_offsets = (
    pd.to_timedelta(hours, unit="h")
    + pd.to_timedelta(minutes, unit="m")
    + pd.to_timedelta(seconds, unit="s")
)

df_bookings_merged["booking_time"] = booking_dates + time_offsets

mask_same_day = days_before == 0

# Random offset between 0–6h before flight
same_day_offsets = pd.to_timedelta(
    np.random.randint(0, 6*60*60, mask_same_day.sum()), unit="s"
)

df_bookings_merged.loc[mask_same_day, "booking_time"] = (
    df_bookings_merged.loc[mask_same_day, "flight_date"] - same_day_offsets
)

df_bookings_merged

,booking_id,customer_id,flight_number,flight_date,booking_time,class_name,frequent_flyer_status_code,price_paid,checked_in,flight_cxl_refund,...,class_bookings,capacity,date_of_birth,nationality,gender,booked_rate,check_in_rate,missed_check_in_drop,age,max_booking_days
0,1,10169503,FL1000000,2022-01-01,2021-12-03 16:23:14,Economy,1-B,34.07,False,False,...,76,130,1994-12-17 21:20:09.821844,Germany,male,58.461538,0.926734,-0.3421,30,75.0
1,2,1407797,FL1133817,2022-01-01,2021-11-13 09:43:34,Economy,1-B,119.0,True,False,...,76,130,1994-11-11 21:02:17.309626,Germany,female,58.461538,0.916740,-0.3321,30,85.0
2,3,12662126,FL1133817,2022-01-01,2021-11-26 07:22:35,Economy,0-N,120.81,True,False,...,76,130,2000-07-19 21:25:02.214333,Germany,male,58.461538,0.890962,-0.3063,25,72.5
3,4,6545603,FL1133817,2022-01-01,2021-12-07 20:37:52,Economy,0-N,120.81,True,False,...,76,130,2004-07-28 21:12:44.491533,Spain,male,58.461538,0.904221,-0.3196,21,70.5
4,5,924261,FL1133817,2022-01-01,2021-11-03 14:35:40,Economy,0-N,120.81,True,False,...,76,130,2004-06-21 21:01:16.661612,United States,female,58.461538,0.894054,-0.3094,21,80.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40633004,40633005,9998191,FL1133669,2024-12-31,2024-10-26 13:22:26,Business,1-B,102.65,True,False,...,12,15,1977-10-13 21:19:45.491733,Finland,female,80.000000,0.986365,-0.1864,47,93.5
40633005,40633006,4962859,FL1133669,2024-12-31,2024-11-01 14:37:01,Business,1-B,102.65,True,False,...,12,15,1975-10-27 21:09:31.459530,United Kingdom,male,80.000000,0.957173,-0.1572,49,84.5
40633006,40633007,7525679,FL1133669,2024-12-31,2024-11-15 10:08:05,Business,0-N,104.21,True,False,...,12,15,1998-08-02 21:14:42.572742,Finland,female,80.000000,0.972473,-0.1725,27,83.5
40633007,40633008,6324710,FL1133669,2024-12-31,2024-12-10 15:03:37,Business,2-S,101.08,True,False,...,12,15,1993-12-05 21:12:17.952529,Germany,male,80.000000,0.968545,-0.1685,31,75.5


## Create Route Cost per Flight Table

Flight costs will be 'reverse engineered':
- we set target profit margins per flight based on distance category and booked rate
- total flight costs are calculated be subtracting flight profits from revenues
- total flight costs are split into various subtotals

In [121]:
df_flights["booked_rate"] = (df_flights["bookings_total"] * 100 / df_flights["seat_capacity"]).round(2)

In [18]:
if csv_settings == "imm":
    df_flights.to_csv(csv_folder_path + "temp_flights.csv", index=False, sep=separator, decimal=dec)

In [122]:
flight_revenues_df = df_bookings[df_bookings["flight_cxl_refund"] == False].groupby("flight_number")["price_paid"].sum().astype(float).rename("flight_rev_total").to_frame()
flight_revenues_df = flight_revenues_df.merge(df_flights[["flight_number", "flight_date", "line_number", "distance_km", "booked_rate"]], on="flight_number")
flight_revenues_df

,flight_number,flight_rev_total,flight_date,line_number,distance_km,booked_rate
0,FL1000000,3804.33,2022-01-01,154,300,62.07
1,FL1000001,4151.91,2022-01-01,186,304,64.83
2,FL1000002,15317.90,2022-01-01,164,1838,55.17
3,FL1000003,8894.64,2022-01-01,101,654,65.52
4,FL1000004,7942.71,2022-01-01,102,431,69.44
...,...,...,...,...,...,...
289339,FL1289339,287149.95,2024-12-31,229,9798,88.89
289340,FL1289340,256754.37,2024-12-31,230,9366,85.19
289341,FL1289341,197794.79,2024-12-31,231,8690,80.76
289342,FL1289342,534827.50,2024-12-31,232,16496,92.74


In [123]:
np.random.seed(50)

short_haul = flight_revenues_df["distance_km"] < 1500
medium_haul = ((flight_revenues_df["distance_km"] >= 1500) &
               (flight_revenues_df["distance_km"] < 5000))
long_haul = (flight_revenues_df["distance_km"] >= 5000)

t1 = flight_revenues_df["booked_rate"] >= 95
t2 = ((flight_revenues_df["booked_rate"] < 95) &
      (flight_revenues_df["booked_rate"] >= 85))
t3 = ((flight_revenues_df["booked_rate"] < 85) &
      (flight_revenues_df["booked_rate"] >= 75))
t4 = ((flight_revenues_df["booked_rate"] < 75) &
      (flight_revenues_df["booked_rate"] >= 70))
t5 = ((flight_revenues_df["booked_rate"] < 70) &
      (flight_revenues_df["booked_rate"] >= 60))
t6 = flight_revenues_df["booked_rate"] < 60

conditions = [
    short_haul & t1,
    short_haul & t2,
    short_haul & t3,
    short_haul & t4,
    short_haul & t5,
    short_haul & t6,

    medium_haul & t1,
    medium_haul & t2,
    medium_haul & t3,
    medium_haul & t4,
    medium_haul & t5,
    medium_haul & t6,

    long_haul & t1,
    long_haul & t2,
    long_haul & t3,
    long_haul & t4,
    long_haul & t5,
    long_haul & t6,
]

profit_margins = [
     # --- Short-haul ---
     np.random.uniform(0.40, 0.55, size=len(flight_revenues_df)),  # t1: 95%+
     np.random.uniform(0.30, 0.40, size=len(flight_revenues_df)),  # t2: 85–94%
     np.random.uniform(0.15, 0.30, size=len(flight_revenues_df)),  # t3: 75–84%
     np.random.uniform(0.05, 0.15, size=len(flight_revenues_df)),  # t4: 70–74%
    -np.random.uniform(0.05, 0.10, size=len(flight_revenues_df)),  # t5: 60–69%
    -np.random.uniform(0.05, 0.15, size=len(flight_revenues_df)),  # t6: <60%

     # --- Medium-haul ---
     np.random.uniform(0.30, 0.45, size=len(flight_revenues_df)),  # t1
     np.random.uniform(0.20, 0.35, size=len(flight_revenues_df)),  # t2
     np.random.uniform(0.10, 0.25, size=len(flight_revenues_df)),  # t3
     np.random.uniform(0.00, 0.10, size=len(flight_revenues_df)),  # t4
    -np.random.uniform(0.05, 0.15, size=len(flight_revenues_df)),  # t5
    -np.random.uniform(0.10, 0.25, size=len(flight_revenues_df)),  # t6

     # --- Long-haul ---
     np.random.uniform(0.25, 0.35, size=len(flight_revenues_df)),  # t1
     np.random.uniform(0.15, 0.25, size=len(flight_revenues_df)),  # t2
     np.random.uniform(0.07, 0.15, size=len(flight_revenues_df)),  # t3
     np.random.uniform(0.01, 0.05, size=len(flight_revenues_df)),  # t4
    -np.random.uniform(0.10, 0.20, size=len(flight_revenues_df)),  # t5
    -np.random.uniform(0.20, 0.40, size=len(flight_revenues_df)),  # t6
]

flight_revenues_df["profit_margin"] = np.select(conditions, profit_margins, default=0.10)
flight_revenues_df

,flight_number,flight_rev_total,flight_date,line_number,distance_km,booked_rate,profit_margin
0,FL1000000,3804.33,2022-01-01,154,300,62.07,-0.083993
1,FL1000001,4151.91,2022-01-01,186,304,64.83,-0.065348
2,FL1000002,15317.90,2022-01-01,164,1838,55.17,-0.201934
3,FL1000003,8894.64,2022-01-01,101,654,65.52,-0.088409
4,FL1000004,7942.71,2022-01-01,102,431,69.44,-0.076017
...,...,...,...,...,...,...,...
289339,FL1289339,287149.95,2024-12-31,229,9798,88.89,0.199227
289340,FL1289340,256754.37,2024-12-31,230,9366,85.19,0.222548
289341,FL1289341,197794.79,2024-12-31,231,8690,80.76,0.112724
289342,FL1289342,534827.50,2024-12-31,232,16496,92.74,0.180087


In [124]:
flight_revenues_df["flight_cost_total"] = (
    flight_revenues_df["flight_rev_total"] * (1 - flight_revenues_df["profit_margin"])
).round(2)

flight_revenues_df["flight_profit_total"] = (
    flight_revenues_df["flight_rev_total"] - flight_revenues_df["flight_cost_total"]
).round(2)

flight_revenues_df["profit_margin_pct"] = (
    100 * flight_revenues_df["flight_profit_total"] / flight_revenues_df["flight_rev_total"]
).round(2)

costs_per_flight_df = flight_revenues_df.copy().drop("profit_margin", axis=1)

costs_per_flight_df

,flight_number,flight_rev_total,flight_date,line_number,distance_km,booked_rate,flight_cost_total,flight_profit_total,profit_margin_pct
0,FL1000000,3804.33,2022-01-01,154,300,62.07,4123.87,-319.54,-8.40
1,FL1000001,4151.91,2022-01-01,186,304,64.83,4423.23,-271.32,-6.53
2,FL1000002,15317.90,2022-01-01,164,1838,55.17,18411.10,-3093.20,-20.19
3,FL1000003,8894.64,2022-01-01,101,654,65.52,9681.00,-786.36,-8.84
4,FL1000004,7942.71,2022-01-01,102,431,69.44,8546.49,-603.78,-7.60
...,...,...,...,...,...,...,...,...,...
289339,FL1289339,287149.95,2024-12-31,229,9798,88.89,229941.80,57208.15,19.92
289340,FL1289340,256754.37,2024-12-31,230,9366,85.19,199614.11,57140.26,22.25
289341,FL1289341,197794.79,2024-12-31,231,8690,80.76,175498.65,22296.14,11.27
289342,FL1289342,534827.50,2024-12-31,232,16496,92.74,438512.20,96315.30,18.01


In [125]:
costs_per_flight_df.groupby("line_number")[["booked_rate", "profit_margin_pct"]].mean().round(2)

,booked_rate,profit_margin_pct
line_number,,
101,72.41,9.02
102,82.37,24.50
103,79.02,19.94
104,78.97,19.91
105,82.46,24.61
...,...,...
238,79.41,9.05
239,90.08,20.67
240,83.09,13.14


In [126]:
df_costs_per_flight = costs_per_flight_df[["flight_number", "flight_date", "flight_cost_total"]].copy()

def generate_cost_split(total):
    fuel = round(total * np.random.uniform(0.32, 0.38), 2)
    crew = round(total * np.random.uniform(0.13, 0.18), 2)
    maintenance = round(total * np.random.uniform(0.10, 0.14), 2)
    landing = round(total * np.random.uniform(0.05, 0.08), 2)
    catering = round(total * np.random.uniform(0.04, 0.06), 2)
    fixed_total = round(fuel + crew + maintenance + landing + catering, 2)
    other = round(total - fixed_total, 2)
    return fuel, crew, maintenance, landing, catering, other

df_costs_per_flight[[
    "fuel_cost", "crew_cost", "maintenance_cost", "landing_fees",
    "catering_cost", "other_costs"
]] = df_costs_per_flight["flight_cost_total"].apply(
    lambda total: pd.Series(generate_cost_split(total))
)

df_costs_per_flight.insert(0, "flight_cost_id", "C_" + df_costs_per_flight["flight_number"].str[2:])

df_costs_per_flight

,flight_cost_id,flight_number,flight_date,flight_cost_total,fuel_cost,crew_cost,maintenance_cost,landing_fees,catering_cost,other_costs
0,C_1000000,FL1000000,2022-01-01,4123.87,1554.05,665.78,569.26,270.80,187.58,876.40
1,C_1000001,FL1000001,2022-01-01,4423.23,1476.71,793.42,598.50,338.32,193.93,1022.35
2,C_1000002,FL1000002,2022-01-01,18411.10,6095.07,2806.25,1883.99,1122.47,863.79,5639.53
3,C_1000003,FL1000003,2022-01-01,9681.00,3589.11,1273.37,1255.22,713.46,415.83,2434.01
4,C_1000004,FL1000004,2022-01-01,8546.49,3227.80,1122.76,1048.53,587.90,457.86,2101.64
...,...,...,...,...,...,...,...,...,...,...
289339,C_1289339,FL1289339,2024-12-31,229941.80,73997.56,36652.99,26656.91,13880.52,10707.64,68046.18
289340,C_1289340,FL1289340,2024-12-31,199614.11,73967.08,32634.50,24447.96,14525.76,9582.12,44456.69
289341,C_1289341,FL1289341,2024-12-31,175498.65,66568.94,24957.75,23799.32,11559.80,7768.41,40844.43
289342,C_1289342,FL1289342,2024-12-31,438512.20,140662.34,70851.23,54902.77,32159.34,22688.31,117248.21


In [19]:
if csv_settings == "imm":
    df_costs_per_flight.to_csv(csv_folder_path + "costs_per_flight.csv", index=False, sep=separator, decimal=dec)

## Finalize Weather, Flights, and Bookings Tables

In [127]:
df_weather_final = df_weather.copy()

df_weather_final["observation_time"] = df_weather_final["observation_time"].dt.tz_localize("UTC")

In [20]:
if csv_settings == "imm":
    df_weather_final.to_csv(csv_folder_path + "weather.csv", index=False, sep=separator, decimal=dec)

In [128]:
# Drop temp columns
df_bookings_final = df_bookings_merged.drop(columns=["date_of_birth", "nationality", "gender", "age", "max_booking_days",
                                                     "check_in_rate", "missed_check_in_drop", "booked_rate", "class_bookings",
                                                     "capacity", "cancellation_reason", "booking_lead_time_days", "booking_hour", "age_group", "booking_hour_window"])
df_bookings_final

,booking_id,customer_id,flight_number,flight_date,booking_time,class_name,frequent_flyer_status_code,price_paid,checked_in,flight_cxl_refund,booking_lead_time_days,booking_hour,age_group,booking_hour_window
0,1,10169503,FL1000000,2022-01-01,2021-12-03 16:23:14,Economy,1-B,34.07,False,False,28,16,(2) Age 25-34,12:00 - 17:00
1,2,1407797,FL1133817,2022-01-01,2021-11-13 09:43:34,Economy,1-B,119.0,True,False,48,9,(2) Age 25-34,6:00 - 12:00
2,3,12662126,FL1133817,2022-01-01,2021-11-26 07:22:35,Economy,0-N,120.81,True,False,35,7,(1) Age <= 24,6:00 - 12:00
3,4,6545603,FL1133817,2022-01-01,2021-12-07 20:37:52,Economy,0-N,120.81,True,False,24,20,(1) Age <= 24,17:00 - 22:00
4,5,924261,FL1133817,2022-01-01,2021-11-03 14:35:40,Economy,0-N,120.81,True,False,58,14,(1) Age <= 24,12:00 - 17:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40633004,40633005,9998191,FL1133669,2024-12-31,2024-10-26 13:22:26,Business,1-B,102.65,True,False,65,13,(4) Age 45-60,12:00 - 17:00
40633005,40633006,4962859,FL1133669,2024-12-31,2024-11-01 14:37:01,Business,1-B,102.65,True,False,59,14,(4) Age 45-60,12:00 - 17:00
40633006,40633007,7525679,FL1133669,2024-12-31,2024-11-15 10:08:05,Business,0-N,104.21,True,False,45,10,(2) Age 25-34,6:00 - 12:00
40633007,40633008,6324710,FL1133669,2024-12-31,2024-12-10 15:03:37,Business,2-S,101.08,True,False,20,15,(2) Age 25-34,12:00 - 17:00


In [129]:
df_flights["bookings_total"].sum() == len(df_bookings_final)

True

In [130]:
df_bookings_final.groupby("flight_number")["checked_in"].count().reset_index().merge(df_flights[["flight_number", "bookings_total"]], on="flight_number", how="left").query("checked_in != bookings_total")

,flight_number,checked_in,bookings_total


In [21]:
if csv_settings == "imm":
    df_bookings_final.to_csv(csv_folder_path + "bookings.csv", index=False, sep=separator, decimal=dec)

In [131]:
df_flights_final = df_flights.copy()

# drop columns "seat_capacity" and "distance_km"; redundant in the final table,
# information available in aircraft and routes tables
df_flights_final = df_flights_final.drop(columns=["seat_capacity", "distance_km", "booked_rate"])

# overwrite "bookings_total" to reflect actual flight passenger counts after cancellations and missed check-ins
df_flights_final["bookings_total"] = df_bookings_final.groupby("flight_number")["checked_in"].sum().values
df_flights_final.rename(columns={"bookings_total": "passengers_total"}, inplace=True)
df_flights_final

,flight_number,flight_date,line_number,aircraft_id,passengers_total,scheduled_departure,scheduled_arrival,actual_departure,actual_arrival,cancelled,cancellation_reason,delay_reason_dep,delay_reason_arr
0,FL1000000,2022-01-01,154,AC1032,82,2022-01-01 01:00:00.000000,2022-01-01 01:31:58.226330,2022-01-01 01:13:00.000000,2022-01-01 01:46:58.226330,False,None,None,None
1,FL1000001,2022-01-01,186,AC1033,86,2022-01-01 01:00:00.000000,2022-01-01 01:35:28.169815,2022-01-01 01:04:00.000000,2022-01-01 01:44:28.169815,False,None,None,None
2,FL1000002,2022-01-01,164,AC1034,75,2022-01-01 01:00:00.000000,2022-01-01 03:16:39.039647,2022-01-01 01:15:00.000000,2022-01-01 03:30:39.039647,False,None,None,None
3,FL1000003,2022-01-01,101,AC1030,89,2022-01-01 01:00:00.000000,2022-01-01 01:59:30.197172,2022-01-01 01:07:00.000000,2022-01-01 02:11:30.197172,False,None,None,None
4,FL1000004,2022-01-01,102,AC1016,0,2022-01-01 01:00:00.000000,2022-01-01 01:37:00.057670,NaT,NaT,True,Technical failure,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
289339,FL1289339,2024-12-31,229,AC3003,235,2024-12-31 23:52:44.012716,2025-01-01 11:00:58.073750,2024-12-31 23:58:44.012716,2025-01-01 11:05:58.073750,False,None,None,None
289340,FL1289340,2024-12-31,230,AC3004,221,2024-12-31 13:04:49.287791,2024-12-31 23:41:50.143004,2024-12-31 13:15:49.287791,2024-12-31 23:54:50.143004,False,None,None,None
289341,FL1289341,2024-12-31,231,AC3005,248,2024-12-31 11:13:02.154258,2024-12-31 21:03:49.757605,2024-12-31 11:39:02.154258,2024-12-31 21:30:49.757605,False,None,Technical issue,Late departure
289342,FL1289342,2024-12-31,232,AC3009,290,2024-12-31 11:47:44.764324,2025-01-01 06:24:49.942215,2024-12-31 11:48:44.764324,2025-01-01 06:26:49.942215,False,None,None,None


In [132]:
df_flights_final.query("cancellation_reason == 'Aircraft unavailable'")

,flight_number,flight_date,line_number,aircraft_id,passengers_total,scheduled_departure,scheduled_arrival,actual_departure,actual_arrival,cancelled,cancellation_reason,delay_reason_dep,delay_reason_arr
12565,FL1012565,2022-04-13,209,AC2011,0,2022-04-13 08:06:31,2022-04-13 09:56:10.582106,NaT,NaT,True,Aircraft unavailable,None,None
13053,FL1013053,2022-04-17,216,AC2008,0,2022-04-17 05:32:00,2022-04-17 09:33:58.860030,NaT,NaT,True,Aircraft unavailable,None,None
13419,FL1013419,2022-04-20,222,AC2001,0,2022-04-20 05:54:44,2022-04-20 09:49:02.247140,NaT,NaT,True,Aircraft unavailable,None,None
14883,FL1014883,2022-05-02,210,AC2011,0,2022-05-02 23:41:31,2022-05-03 02:49:27.349081,NaT,NaT,True,Aircraft unavailable,None,None
267683,FL1267683,2022-01-13,239,AC3000,0,2022-01-13 11:20:37,2022-01-13 22:29:38.326841,NaT,NaT,True,Aircraft unavailable,None,None
268563,FL1268563,2022-02-26,240,AC3003,0,2022-02-26 17:20:19,2022-02-27 03:51:26.150565,NaT,NaT,True,Aircraft unavailable,None,None
268583,FL1268583,2022-02-27,241,AC3011,0,2022-02-27 16:58:35,2022-02-28 02:53:21.107133,NaT,NaT,True,Aircraft unavailable,None,None
280443,FL1280443,2023-10-13,236,AC3005,0,2023-10-13 02:46:14,2023-10-13 13:09:16.566395,NaT,NaT,True,Aircraft unavailable,None,None


In [133]:
df_flights_final.merge(df_aircraft[["aircraft_id", "seat_capacity"]], on="aircraft_id").query("passengers_total > seat_capacity")

,flight_number,flight_date,line_number,aircraft_id,passengers_total,scheduled_departure,scheduled_arrival,actual_departure,actual_arrival,cancelled,cancellation_reason,delay_reason_dep,delay_reason_arr,seat_capacity


In [22]:
if csv_settings == "imm":
    df_flights_final.to_csv(csv_folder_path + "flights.csv", index=False, sep=separator, decimal=dec)

## Create Frequent Flyer Discounts Table

In [134]:
# Add 1 to thresholds because status mapping uses "<=", so actual minimums are x+1
yearly_flights_min = [pd.NA] + [x + 1 for x in thresholds]

discount_data = []
for code, name, flights, discount in zip(
        list(discounts.keys()),
        ["No Status", "Basic", "Silver", "Gold", "Platinum"],
        yearly_flights_min,
        list(discounts.values())):

    discount_data.append({"frequent_flyer_status_code": code,
                          "frequent_flyer_status": name,
                          "min_flights_yearly": flights,
                          "frequent_flyer_discount_pct": discount})

df_frequent_flyer_discounts = pd.DataFrame(discount_data)
df_frequent_flyer_discounts

,frequent_flyer_status_code,frequent_flyer_status,min_flights_yearly,frequent_flyer_discount_pct
0,0-N,None,<NA>,0.0
1,1-B,Basic,4,1.5
2,2-S,Silver,6,3.0
3,3-G,Gold,8,8.0
4,4-P,Platinum,11,13.0


In [204]:
frequent_flyer_discounts = df_frequent_flyer_discounts.copy()

# for potential use in Excel/Power BI: divide pct column by 100 since application of
#   percentage format in these apps will automatically multiply by 100
frequent_flyer_discounts["frequent_flyer_discount_pct"] = frequent_flyer_discounts["frequent_flyer_discount_pct"] / 100
frequent_flyer_discounts.rename(columns={"frequent_flyer_discount_pct": "frequent_flyer_discount"}, inplace=True)

if csv_settings == "imm":
    frequent_flyer_discounts.to_csv(csv_folder_path + "frequent_flyer_discounts.csv", index=False, sep=separator, decimal=dec)

## Add Noise to Customers Table

#### 🧹 Possible Cleaning Tasks
- remove or fill in missing values

- normalize name casing and country names

- validate and fix emails

In [5]:
# Introduce noise settings
MISSING_RATE = 0.02     # 2% of rows will have missing data
TYPO_RATE = 0.01         # 1% will have email/name/nationality typos

random.seed(50)

# Helper function to introduce email typos
def add_email_typo(email_):
    username, sep, domain = email_.partition("@")
    if not sep:  # No "@" found
        return email_

    typo_options = [
        lambda u, d: u + "@gnail.com",
        lambda u, d: u + "@gmal.com",
        lambda u, d: u + "@yaho.com",
        lambda u, d: u + "@hotmial.com",
        lambda u, d: u + "@gmail..com",
        lambda u, d: u + "@yahoo..com",
        lambda u, d: u + "@hotmail..com",
        lambda u, d: u + "@@gmail.com",
        lambda u, d: u + "@@yahoo.com",
        lambda u, d: u + "@@hotmail.com",
        lambda u, d: u + "@gmail.comcom",
        lambda u, d: u + "@yahoo.comcom",
        lambda u, d: u + "@hotmail.comcom"
    ]
    typo_func = random.choice(typo_options)
    return typo_func(username, domain)

# Create noisy dataset
noisy_data = []

for row in df_customers.itertuples(index=False):
    customer_id, full_name, email, phone, dob, nationality, gender, status_ = row

    # Possibly introduce typos
    if random.random() < TYPO_RATE:
        if random.random() < 0.5:
            email = add_email_typo(email)
        else:
            full_name = full_name.lower()  # messy casing

    # Possibly lowercase nationality (format issue)
    if random.random() < TYPO_RATE:
        nationality = nationality.lower()

    # Possibly introduce missing fields
    if random.random() < MISSING_RATE:
        email = None if random.random() < 0.5 else email  # 50% chance if triggered
    if random.random() < MISSING_RATE:
        phone = None if random.random() < 0.5 else phone
    if random.random() < MISSING_RATE:
        dob = None if random.random() < 0.5 else dob

    noisy_data.append((customer_id, full_name, email, phone, dob, nationality, gender, status_))

df_customers_noisy = pd.DataFrame(noisy_data, columns=[
    "customer_id", "full_name", "email", "phone", "date_of_birth", "nationality", "gender", "frequent_flyer_status_code"
])

df_customers_noisy

,customer_id,full_name,email,phone,date_of_birth,nationality,gender,frequent_flyer_status_code
0,1,Zoé Da Costa,zoe.dacosta@yahoo.com,1.737752e+09,2002-10-03 20:59:15.975115,France,female,0-N
1,2,Jamie Barrett,jamie.barrett@yahoo.com,1.896312e+09,1999-02-01 20:59:15.982120,United Kingdom,male,0-N
2,3,Cornelio Salas,cornelio.salas@yahoo.com,1.397925e+07,1985-10-05 20:59:15.982120,Spain,male,0-N
3,4,Heinrich Plath,heinrich.plath@hotmail.com,1.827998e+09,1990-09-20 20:59:15.983624,Germany,male,0-N
4,5,Zita Zobel,zita.zobel@hotmail.com,1.732291e+09,2000-07-18 20:59:15.983624,Germany,female,0-N
...,...,...,...,...,...,...,...,...
13087524,13815218,Andrew Metcalfe,andrew.metcalfe@hotmail.com,6.536034e+09,1963-05-29 21:27:25.758252,United Kingdom,male,1-B
13087525,13815219,Araceli Amor,araceli.amor@yahoo.com,1.560689e+09,1992-01-29 21:27:25.758252,Spain,female,1-B
13087526,13815220,Damien Jones,damien.jones@gmail.com,2.068736e+09,1995-11-21 21:27:25.758252,United Kingdom,male,0-N
13087527,13815221,Gerold Lange,gerold.lange@yahoo.com,1.296954e+09,1991-05-14 21:27:25.758252,Germany,male,0-N


In [6]:
g_nail_mask = [str(x).endswith("@gnail.com") for x in df_customers_noisy["email"]]
df_customers_noisy.loc[g_nail_mask, "email"]

8813        zdravko.schomber@gnail.com
11948          morten.jensen@gnail.com
13425          patrick.smith@gnail.com
15753              max.james@gnail.com
21334         rebecca.binner@gnail.com
                       ...            
13071196       karen.vasquez@gnail.com
13071739     francoise.bloch@gnail.com
13072097       lili.schuster@gnail.com
13073909       andrew.barlow@gnail.com
13077340      william.pearce@gnail.com
Name: email, Length: 4963, dtype: object

In [23]:
if csv_settings == "imm":
    df_customers_noisy.to_csv(csv_folder_path + "customers_noisy.csv", index=False, sep=separator, decimal=dec)

## All CSVs (/xlxs)

In [137]:
if csv_settings == "end":
    for table_name, table in {
                              "airports": df_airports,
                              "aircraft": df_aircraft,
                              "costs_per_flight": df_costs_per_flight,
                              "routes": df_routes,
                              "weather": df_weather_final,
                              "flights": df_flights_final,
                              "flight_capacity_by_class": df_flight_capacity_by_class,
                              "flight_class_cost_shares": df_flight_class_cost_shares,
                              "bookings": df_bookings_final,
                              "frequent_flyer_discounts": frequent_flyer_discounts,
                              "customers_before_noise": df_customers,
                              "customers_noisy": df_customers_noisy,
                              }.items():
        table.to_csv(csv_folder_path + table_name + ".csv", index=False, sep=separator, decimal=dec)

In [28]:
if simulation_output == "excel":
    airline_data = {
                    "airports": df_airports,
                    "aircraft": df_aircraft,
                    "costs_per_flight": df_costs_per_flight,
                    "routes": df_routes,
                    "weather": df_weather,
                    "flights": df_flights_final,
                    "flight_capacity_by_class": df_flight_capacity_by_class,
                    "flight_class_cost_shares": df_flight_class_cost_shares,
                    "bookings": df_bookings_final,
                    "frequent_flyer_discounts": df_frequent_flyer_discounts,
                    "customers": df_customers,
                  }

    writer = pd.ExcelWriter(xlsx_folder_path + "airline_data.xlsx", engine="xlsxwriter")

    for sheet_name, sheet in airline_data.items():
       sheet.to_excel(writer, sheet_name=sheet_name, index=False)

    writer.close()

## Possible Additions

In [7]:
"""
-- PAYMENTS
CREATE TABLE payments (
    payment_id SERIAL PRIMARY KEY,
    booking_id INT,
    payment_time TIMESTAMP,
    amount DECIMAL(8,2),
    payment_method VARCHAR(30),
    status VARCHAR(20),
    FOREIGN KEY (booking_id) REFERENCES bookings(booking_id)
);

-- MAINTENANCE RECORDS
CREATE TABLE maintenance_records (
    maintenance_id SERIAL PRIMARY KEY,
    aircraft_id VARCHAR(10),
    maintenance_date DATE,
    maintenance_type VARCHAR(100),
    notes TEXT,
    FOREIGN KEY (aircraft_id) REFERENCES aircraft(aircraft_id)
);

-- LOYALTY PROGRAM
CREATE TABLE loyalty_program (
    customer_id INT PRIMARY KEY,
    frequent_flyer_status_code VARCHAR(10),
    points_balance INT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (frequent_flyer_status_code) REFERENCES frequent_flyer_discounts(frequent_flyer_status_code)
);
"""

'\n-- PAYMENTS\nCREATE TABLE payments (\n    payment_id SERIAL PRIMARY KEY,\n    booking_id INT,\n    payment_time TIMESTAMP,\n    amount DECIMAL(8,2),\n    payment_method VARCHAR(30),\n    status VARCHAR(20),\n    FOREIGN KEY (booking_id) REFERENCES bookings(booking_id)\n);\n\n-- MAINTENANCE RECORDS\nCREATE TABLE maintenance_records (\n    maintenance_id SERIAL PRIMARY KEY,\n    aircraft_id VARCHAR(10),\n    maintenance_date DATE,\n    maintenance_type VARCHAR(100),\n    notes TEXT,\n    FOREIGN KEY (aircraft_id) REFERENCES aircraft(aircraft_id)\n);\n\n-- LOYALTY PROGRAM\nCREATE TABLE loyalty_program (\n    customer_id INT PRIMARY KEY,\n    tier VARCHAR(20),\n    points_balance INT,\n    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)\n);\n'

# Load Airline Data into SQL Tables

In [31]:
# connect to the airline_data database, created with default settings via preferred SQL admin tool or terminal
# urllib.parse only required for pw with special characters
def connect_to_db():
    with open("C:/Users/Default/pstgr_pw.txt", "r") as f:
        db_pw = parse.quote(f.read())
    db_user = 'postgres'
    db_host = 'localhost'
    db_port = '5432'
    db_name = 'airline_data'

    engine_ = create_engine(f'postgresql://{db_user}:{db_pw}@{db_host}:{db_port}/{db_name}')
    return engine_

In [ ]:
# create SQL-database tables
def create_tables():
    with engine.begin() as conn_:
        conn_.execute(text("""

            CREATE TABLE airports (
                airport_code CHAR(3) PRIMARY KEY,
                airport_name VARCHAR(100),
                city VARCHAR(50),
                country VARCHAR(50),
                coordinates POINT,
                climate_region VARCHAR(50),
                hub_status VARCHAR(20)
             );

            CREATE TABLE aircraft(
                aircraft_id      CHAR(10) PRIMARY KEY,
                model            VARCHAR(50),
                manufacturer     VARCHAR(50),
                seat_capacity    INT,
                range_km         INT,
                manufacture_year INT
             );

            CREATE TABLE routes (
                line_number SERIAL PRIMARY KEY,
                departure_airport_code CHAR(3),
                arrival_airport_code CHAR(3),
                distance_km INT,
                flights_per_day INT,
                price_economy DECIMAL(8,2),
                price_business DECIMAL(8,2),
                price_first DECIMAL(8,2),
                FOREIGN KEY (departure_airport_code) REFERENCES airports(airport_code),
                FOREIGN KEY (arrival_airport_code) REFERENCES airports(airport_code),
                CHECK (departure_airport_code != arrival_airport_code)
             );

            CREATE TABLE weather (
                weather_id SERIAL PRIMARY KEY,
                airport_code CHAR(3),
                observation_time TIMESTAMPTZ,
                season VARCHAR(10),
                temperature_celsius DECIMAL(4,1),
                wind_speed_kmh DECIMAL(5,1),
                visibility_km DECIMAL(4,1),
                condition_description VARCHAR(100),
                FOREIGN KEY (airport_code) REFERENCES airports(airport_code)
             );

            CREATE TABLE flights (
                flight_number VARCHAR(10),
                flight_date DATE,
                line_number INT,
                aircraft_id VARCHAR(10),
                passengers_total INT,
                scheduled_departure TIMESTAMP,
                scheduled_arrival TIMESTAMP,
                actual_departure TIMESTAMP,
                actual_arrival TIMESTAMP,
                cancelled BOOLEAN,
                cancellation_reason TEXT,
                delay_reason_dep TEXT,
                delay_reason_arr TEXT,
                PRIMARY KEY (flight_number, flight_date),
                FOREIGN KEY (line_number) REFERENCES routes(line_number),
                FOREIGN KEY (aircraft_id) REFERENCES aircraft(aircraft_id)
             );

            CREATE TABLE costs_per_flight (
                flight_cost_id VARCHAR(14),
                flight_number VARCHAR(10),
                flight_date DATE,
                flight_cost_total DECIMAL(10,2),
                fuel_cost DECIMAL(10,2),
                crew_cost DECIMAL(10,2),
                maintenance_cost DECIMAL(10,2),
                landing_fees DECIMAL(10,2),
                catering_cost DECIMAL(10,2),
                other_costs DECIMAL(10,2),
                PRIMARY KEY (flight_cost_id),
                FOREIGN KEY (flight_number, flight_date) REFERENCES flights(flight_number, flight_date)
             );

            CREATE TABLE flight_capacity_by_class (
                flight_number VARCHAR(10),
                flight_date DATE,
                class_name VARCHAR(20),
                capacity INT,
                class_bookings INT,
                PRIMARY KEY (flight_number, flight_date, class_name),
                FOREIGN KEY (flight_number, flight_date) REFERENCES flights(flight_number, flight_date),
                CHECK (class_bookings <= capacity)
             );

            CREATE TABLE flight_class_cost_shares (
                cost_share_id VARCHAR(14),
                flight_number VARCHAR(10),
                flight_date DATE,
                class_name VARCHAR(20),
                cost_share NUMERIC(4,2),
                PRIMARY KEY (cost_share_id),
                FOREIGN KEY (flight_number, flight_date) REFERENCES flights(flight_number, flight_date),
                CHECK (cost_share >= 0 AND cost_share <= 1)
             );

            CREATE TABLE frequent_flyer_discounts (
                frequent_flyer_status_code VARCHAR(10),
                frequent_flyer_status VARCHAR(20),
                min_flights_yearly INT,
                frequent_flyer_discount_pct DECIMAL(5,1),
                PRIMARY KEY (frequent_flyer_status_code)
             );

            CREATE TABLE customers (
                customer_id SERIAL PRIMARY KEY,
                full_name VARCHAR(100),
                email VARCHAR(100),
                phone VARCHAR(100),
                date_of_birth DATE,
                nationality VARCHAR(100),
                gender VARCHAR(10),
                frequent_flyer_status_code VARCHAR(10),
                FOREIGN KEY (frequent_flyer_status_code) REFERENCES frequent_flyer_discounts(frequent_flyer_status_code)
             );

            CREATE TABLE bookings (
                booking_id SERIAL,
                customer_id INT,
                flight_number VARCHAR(10),
                flight_date DATE NOT NULL,
                booking_time TIMESTAMP,
                class_name VARCHAR(20),
                frequent_flyer_status_code VARCHAR(10),
                price_paid DECIMAL(8,2),
                checked_in BOOLEAN,
                flight_cxl_refund BOOLEAN DEFAULT FALSE,
                PRIMARY KEY (booking_id, flight_date),
                FOREIGN KEY (flight_number, flight_date) REFERENCES flights(flight_number, flight_date),
                FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
                FOREIGN KEY (frequent_flyer_status_code) REFERENCES frequent_flyer_discounts(frequent_flyer_status_code)
            ) PARTITION BY RANGE (flight_date);

            CREATE TABLE bookings_2022 (
                booking_id SERIAL,
                customer_id INT,
                flight_number VARCHAR(10),
                flight_date DATE NOT NULL,
                booking_time TIMESTAMP,
                class_name VARCHAR(20),
                frequent_flyer_status_code VARCHAR(10),
                price_paid DECIMAL(8,2),
                checked_in BOOLEAN,
                flight_cxl_refund BOOLEAN DEFAULT FALSE
             );
            CREATE TABLE bookings_2023 (
                booking_id SERIAL,
                customer_id INT,
                flight_number VARCHAR(10),
                flight_date DATE NOT NULL,
                booking_time TIMESTAMP,
                class_name VARCHAR(20),
                frequent_flyer_status_code VARCHAR(10),
                price_paid DECIMAL(8,2),
                checked_in BOOLEAN,
                flight_cxl_refund BOOLEAN DEFAULT FALSE
             );
            CREATE TABLE bookings_2024 (
                booking_id SERIAL,
                customer_id INT,
                flight_number VARCHAR(10),
                flight_date DATE NOT NULL,
                booking_time TIMESTAMP,
                class_name VARCHAR(20),
                frequent_flyer_status_code VARCHAR(10),
                price_paid DECIMAL(8,2),
                checked_in BOOLEAN,
                flight_cxl_refund BOOLEAN DEFAULT FALSE
             );

             """)
    )

### Loading Option I: Directly from Dataframes

In [33]:
engine = connect_to_db()
if database_tables == "here":
    create_tables()

In [139]:
for table_name, df in {"airports": df_airports, "aircraft": df_aircraft, "routes": df_routes, "frequent_flyer_discounts": df_frequent_flyer_discounts}.items():
    df.to_sql(table_name, engine, if_exists="append", index=False)

In [140]:
df_weather_final.to_sql("weather", engine, if_exists="append", index=False)

768

In [141]:
df_flights_final.to_sql("flights", engine, if_exists="append", index=False)

344

In [142]:
df_costs_per_flight.to_sql("costs_per_flight", engine, if_exists="append", index=False)

344

In [143]:
df_flight_capacity_by_class.to_sql("flight_capacity_by_class", engine, if_exists="append", index=False)

32

In [144]:
df_flight_class_cost_shares.to_sql("flight_class_cost_shares", engine, if_exists="append", index=False)

774

In [145]:
b1 = df_bookings_final[df_bookings_final["flight_date"].dt.year == 2022]
b2 = df_bookings_final[df_bookings_final["flight_date"].dt.year == 2023]
b3 = df_bookings_final[df_bookings_final["flight_date"].dt.year == 2024]

In [146]:
df_bookings_final["flight_date"].dt.year.value_counts().sort_index()

flight_date
2022    13540763
2023    13532672
2024    13559574
Name: count, dtype: int64

In [147]:
len(df_bookings_final), (len(b1), len(b2), len(b3)), len(df_bookings) == sum((len(b1), len(b2), len(b3)))

(40633009, (13540763, 13532672, 13559574), True)

In [ ]:
# use df_customers instead of df_customers_noisy if no noise wanted
df_customers_noisy.to_sql("customers", engine, if_exists="append", index=False)

In [261]:
b1.to_sql("bookings_2022", engine, if_exists="append", index=False)
b2.to_sql("bookings_2023", engine, if_exists="append", index=False)
b3.to_sql("bookings_2024", engine, if_exists="append", index=False)

503

In [ ]:
# convert yearly booking tables to partitions of bookings (after all previous tables have been loaded)
def attach_partitions():
    with engine.begin() as conn:
        conn.execute(text("""

        ALTER TABLE bookings
            ATTACH PARTITION bookings_2022
            FOR VALUES FROM ('2022-01-01') TO ('2023-01-01');
        ALTER TABLE bookings
            ATTACH PARTITION bookings_2023
            FOR VALUES FROM ('2023-01-01') TO ('2024-01-01');
        ALTER TABLE bookings
            ATTACH PARTITION bookings_2024
            FOR VALUES FROM ('2024-01-01') TO ('2025-01-01');

        """)
    )

if database_tables == "here":
    attach_partitions()

### Loading Option II: from CSVs

In [ ]:
load_from_csvs = False    # set to True if needed

In [ ]:
engine = connect_to_db()
if database_tables == "here":
    create_tables()

In [ ]:
if load_from_csvs:
    csv_files = {}

    for table_name in ["airports", "aircraft", "route_cost_per_flight", "routes"]:
        csv_files[table_name] = f"{csv_folder_path}{table_name}.csv"

    for table_name, file_path in csv_files.items():
        df = pd.read_csv(file_path, sep=separator, decimal=dec)
        df.to_sql(table_name, engine, if_exists='append', index=False)
        print(f"Inserted data into table: {table_name}")

Tables with datetime columns should be read in with parse_dates; recommended: execute the loading of large tables in separate cells

In [ ]:
if load_from_csvs:
    frequent_flyer_discounts_df = pd.read_csv(f"{csv_folder_path}frequent_flyer_discounts.csv", sep=separator, decimal=dec)
    frequent_flyer_discounts_df["frequent_flyer_discount_pct"] = df_frequent_flyer_discounts["frequent_flyer_discount_pct"] * 100
    frequent_flyer_discounts_df.rename(columns={"frequent_flyer_discount": "frequent_flyer_discount_pct"}, inplace=True)

    frequent_flyer_discounts_df.to_sql("frequent_flyer_discounts", engine, if_exists="append", index=False)

In [ ]:
if load_from_csvs:
    weather_df_ = pd.read_csv(f"{csv_folder_path}weather.csv", parse_dates=["observation_time"], sep=separator, decimal=dec)
    weather_df_.to_sql("weather", engine, if_exists="append", index=False)

In [ ]:
if load_from_csvs:
    flights_df = (
        pd.read_csv(f"{csv_folder_path}flights.csv",
                    parse_dates=["flight_date", "scheduled_departure", "scheduled_arrival", "actual_departure", "actual_arrival"],
                    sep=separator, decimal=dec))
    flights_df.to_sql("flights", engine, if_exists="append", index=False)

In [ ]:
if load_from_csvs:
    costs_per_flight_df = pd.read_csv(f"{csv_folder_path}costs_per_flight.csv", parse_dates=["flight_date"], sep=separator, decimal=dec)
    costs_per_flight_df.to_sql("costs_per_flight", engine, if_exists="append", index=False)

In [ ]:
if load_from_csvs:
    flight_capacity_by_class_df = pd.read_csv(f"{csv_folder_path}flight_capacity_by_class.csv", parse_dates=["flight_date"], sep=separator, decimal=dec)
    flight_capacity_by_class_df.to_sql("flight_capacity_by_class", engine, if_exists="append", index=False)

In [ ]:
if load_from_csvs:
    flight_class_cost_shares_df = pd.read_csv(f"{csv_folder_path}flight_class_cost_shares.csv", sep=separator, decimal=dec)
    flight_class_cost_shares_df.to_sql("flight_class_cost_shares", engine, if_exists="append", index=False)

In [ ]:
if load_from_csvs:
    customers_df = pd.read_csv(f"{csv_folder_path}customers.csv", parse_dates=["date_of_birth"], dtype={"phone": "object"}, sep=separator, decimal=dec)
    customers_df.to_sql("customers", engine, if_exists='append', index=False)

In [ ]:
if load_from_csvs:
    bookings_df = pd.read_csv(f"{csv_folder_path}bookings.csv", parse_dates=["flight_date", "booking_time"], sep=separator, decimal=dec)

    # if partitioned (adjust dates where necessary):
    b1 = bookings_df[bookings_df["flight_date"].dt.year == 2022]
    b2 = bookings_df[bookings_df["flight_date"].dt.year == 2023]
    b3 = bookings_df[bookings_df["flight_date"].dt.year == 2024]

    print(
        bookings_df["flight_date"].dt.year.value_counts().sort_index(), "\n",
        len(bookings_df), (len(b1), len(b2), len(b3)), len(bookings_df) == sum((len(b1), len(b2), len(b3)))
        )

In [ ]:
if load_from_csvs:
    b1.to_sql("bookings_2022", engine, if_exists="append", index=False)
    b2.to_sql("bookings_2023", engine, if_exists="append", index=False)
    b3.to_sql("bookings_2024", engine, if_exists="append", index=False)

In [ ]:
if load_from_csvs:
    if database_tables == "here":
        attach_partitions()

### Example: Updating Database

In [108]:
'''
flights_updates = df_flights_final[["flight_number", "cancellation_reason", "delay_reason"]]
flights_updates.to_sql("temp_flights_updates", engine, if_exists="replace", index=False)

engine = connect_to_db()
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE flights
        SET
            cancellation_reason = temp_flights_updates.cancellation_reason,
            delay_reason = temp_flights_updates.delay_reason
        FROM temp_flights_updates
        WHERE flights.flight_number = temp_flights_updates.flight_number
    """))
'''